In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import calendar

from scipy import stats
import statsmodels.api as sm
from scipy import stats
import statsmodels.formula.api as smf

import scipy.stats as scipy_stats

## Data Load

In [4]:
def wrangle(df: pd.DataFrame) -> pd.DataFrame:
    df.rename(
        columns={
            "data_type": "data_type",
            "source": "provider",
            "Sales Date": "date",
            "Outlet SF ID": "customer_code",
            "Store Participant Code": "customer_name",
            "SKU SF ID": "sku_code",
            "SKU Name": "sku_name",
            "Brand Variant": "brand_variant",
            "Brand Family": "brand_name",
            "Category": "category",
            "Volume in Unit": "sales_amount",
            "Volume in Packs": "sales_quantity",
            "Ownership Type": "channel_name",
            "Latitude": "latitude",
            "Longitude": "longitude",
            "Territory Id": "route",
            # Extra
            "Brand": "brand",
            "SKU Clean": "sku_clean",
            "Month": "month",
        },
        inplace=True,
    )

    df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")

    return df


def load_sell_data(df_path):
    try:
        df = pd.read_csv(df_path, compression="gzip", low_memory=False)
        df = wrangle(df)
        df_sellin = df[df["data_type"] == "sell_in"].copy()
        df_sellout = df[
            (df["data_type"] == "sell_out")
            & (df["sku_code"].astype(str).str.strip() != "0")
        ].copy()
        # Replace missing category with FMC
        df_sellout["category"] = df_sellout["category"].fillna("FMC")
        return df_sellin, df_sellout
    except FileNotFoundError:
        print(
            "Could not find `combined_df.csv`. Make sure it is in the same folder as `app.py`."
        )
        return pd.DataFrame()

In [5]:
df_path = "/home/asad/Downloads/combined_df.gz"
weather_path = "/home/asad/Desktop/footfall_explorer/data/France/processed_data/weather_2024_2026may.csv"

In [6]:
sellin, sellout = load_sell_data(df_path)
weather_df = pd.read_csv(weather_path, low_memory=False)

weather_df["date"] = pd.to_datetime(weather_df["date"], errors="coerce").dt.normalize()

## Exploration of STL

### Explore Single Shop

In [7]:
selected_customer_data = sellout[
    sellout["customer_code"] == "0011t000011b2hEAAQ"
].copy()
print(selected_customer_data.shape)
selected_customer_data.drop_duplicates(inplace=True)
print(selected_customer_data.shape)
new_data = (
    selected_customer_data.groupby("date")
    .agg(
        {
            "sales_quantity": "sum",
            "sales_amount": "sum",
            "latitude": "first",
            "longitude": "first",
        }
    )
    .reset_index()
)

(31979, 20)
(31841, 20)


In [8]:
merged_data = new_data.merge(
    weather_df,
    on=["date", "latitude", "longitude"],
    how="left",  # change to inner/right if needed
)

In [9]:
fig = px.line(
    merged_data, x="date", y="sales_quantity", title="Daily Volume Packs", markers=True
)

fig.update_layout(xaxis_title="Date", yaxis_title="Volume Packs", hovermode="x unified")

fig.show()

In [10]:
m = merged_data.copy()

d = m[m["sales_quantity"] > 0].copy()  # drop closed-ish days for a fair test
d["rained"] = d["precipitation"] > 3

# 1. raw correlations
pear = d[["sales_quantity", "precipitation"]].corr().iloc[0, 1]
spear = d[["sales_quantity", "precipitation"]].corr("spearman").iloc[0, 1]
print(f"Pearson  qty~precip: {pear:+.3f}")
print(f"Spearman qty~precip: {spear:+.3f}  (rank-based, less outlier-sensitive)")

# 2. dry vs rainy, with a t-test
dry = d.loc[~d["rained"], "sales_quantity"]
rain = d.loc[d["rained"], "sales_quantity"]
t, p = stats.ttest_ind(dry, rain, equal_var=False)
print(
    f"\nMean qty  dry={dry.mean():.1f} (n={len(dry)})  "
    f"rainy={rain.mean():.1f} (n={len(rain)})"
)
print(f"Difference: {(rain.mean()-dry.mean())/dry.mean():+.1%}   " f"t-test p={p:.3f}")

# 3. remove weekday pattern, then re-test (weekend != weekday, rain or not)
d["dow"] = d["date"].dt.dayofweek
d["resid"] = d["sales_quantity"] - d.groupby("dow")["sales_quantity"].transform("mean")
t2, p2 = stats.ttest_ind(
    d.loc[~d["rained"], "resid"], d.loc[d["rained"], "resid"], equal_var=False
)
print(f"\nAfter removing weekday effect:")
print(
    f"  rainy-day deviation: {d.loc[d['rained'],'resid'].mean():+.1f} units"
    f"   p={p2:.3f}"
)

Pearson  qty~precip: -0.029
Spearman qty~precip: -0.031  (rank-based, less outlier-sensitive)

Mean qty  dry=159.5 (n=600)  rainy=158.9 (n=213)
Difference: -0.3%   t-test p=0.892

After removing weekday effect:
  rainy-day deviation: -1.2 units   p=0.666


In [11]:
RAIN_MM = 3  # single definition of a "rain day"
m = merged_data.copy()
m["rained"] = m["precipitation"] > RAIN_MM

# --- STL needs a gap-free daily series; m has missing (closed) days ---
s = m.set_index("date")["sales_quantity"].asfreq("D")
print("days after asfreq:", len(s), " missing:", s.isna().sum())
s = s.interpolate("time", limit_direction="both")  # bridge closed-day gaps for STL only

stl = sm.tsa.STL(s, period=7, robust=True).fit()

dec = pd.DataFrame(
    {
        "trend": stl.trend,
        "seasonal": stl.seasonal,
        "remainder": stl.resid,
    }
)
dec.index.name = "date"  # the STL components are indexed by date
dec = dec.reset_index()  # now 'date' is a plain column, no ambiguity
dec = dec.merge(
    m[["date", "precipitation", "rained", "sales_quantity"]], on="date", how="inner"
)

# keep only real selling days (sales_quantity > 0)
real_days = m.loc[m["sales_quantity"] > 0, "date"]
dec = dec[dec["date"].isin(real_days)]

# (1) does rain explain the STL remainder?  -- continuous
r = dec[["remainder", "precipitation"]].corr().iloc[0, 1]
t, p = stats.ttest_ind(
    dec.loc[~dec["rained"], "remainder"],
    dec.loc[dec["rained"], "remainder"],
    equal_var=False,
)
print(f"\nRemainder ~ precip  corr={r:+.3f}")
print(
    f"Remainder: dry mean={dec.loc[~dec['rained'],'remainder'].mean():+.1f}  "
    f"rainy mean={dec.loc[dec['rained'],'remainder'].mean():+.1f}  p={p:.3f}"
)

# (2) rainfall in BANDS, not a single slope -- this answers the non-linearity question
dec["band"] = pd.cut(
    dec["precipitation"],
    [-0.01, 0.1, 2, 8, 1e9],
    labels=["none", "light", "moderate", "heavy"],
)
print("\nMean STL remainder by rainfall band (0 = an average day):")
print(
    dec.groupby("band", observed=True)["remainder"]
    .agg(["count", "mean"])
    .round(2)
    .to_string()
)

# (3) is the band trend statistically distinguishable?  ANOVA across bands
groups = [g["remainder"].values for _, g in dec.groupby("band", observed=True)]
F, pa = stats.f_oneway(*groups)
print(f"\nANOVA across bands: F={F:.2f}  p={pa:.3f}")

days after asfreq: 820  missing: 7

Remainder ~ precip  corr=-0.075
Remainder: dry mean=+1.9  rainy mean=+0.4  p=0.452

Mean STL remainder by rainfall band (0 = an average day):
          count  mean
band                 
none        362  2.11
light       197  2.03
moderate    171  1.36
heavy        83 -2.44

ANOVA across bands: F=0.76  p=0.516


In [12]:
band_stats = (
    dec.groupby("band", observed=True)["remainder"]
    .agg(["count", "mean", "std"])
    .reset_index()
)
band_stats["sem"] = band_stats["std"] / band_stats["count"] ** 0.5

fig = make_subplots(
    rows=1,
    cols=2,
    column_widths=[0.42, 0.58],
    subplot_titles=(
        "Mean STL remainder by rainfall band",
        "Remainder vs precipitation (each dot = one day)",
    ),
)

# --- left: band bars with error bars + day counts on hover ---
colors = ["#9aa5b1", "#5b9bd5", "#2f6da8", "#c0392b"]
fig.add_trace(
    go.Bar(
        x=band_stats["band"],
        y=band_stats["mean"],
        error_y=dict(type="data", array=band_stats["sem"], visible=True),
        marker_color=colors,
        customdata=band_stats[["count", "std"]],
        hovertemplate=(
            "<b>%{x}</b><br>mean remainder: %{y:.2f} units"
            "<br>days: %{customdata[0]}"
            "<br>std: %{customdata[1]:.1f}<extra></extra>"
        ),
        showlegend=False,
    ),
    row=1,
    col=1,
)
fig.add_hline(y=0, line_dash="dot", line_color="gray", row=1, col=1)

# --- right: raw scatter, colored by band, with a 0 reference line ---
for b, c in zip(["none", "light", "moderate", "heavy"], colors):
    sub = dec[dec["band"] == b]
    fig.add_trace(
        go.Scatter(
            x=sub["precipitation"],
            y=sub["remainder"],
            mode="markers",
            name=b,
            marker=dict(color=c, size=6, opacity=0.6),
            hovertemplate=(
                f"<b>{b}</b><br>precip: %{{x:.1f}} mm"
                "<br>remainder: %{y:.1f} units<extra></extra>"
            ),
        ),
        row=1,
        col=2,
    )
fig.add_hline(y=0, line_dash="dot", line_color="gray", row=1, col=2)

fig.update_xaxes(title_text="rainfall band", row=1, col=1)
fig.update_yaxes(title_text="STL remainder (units vs expected)", row=1, col=1)
fig.update_xaxes(title_text="precipitation (mm)", row=1, col=2)
fig.update_layout(
    height=460,
    width=1000,
    title_text="Rain vs sales — single shop (STL remainder)",
    template="plotly_white",
)
fig.show()

In [13]:
# Plot
resid_colors = ["#27ae60" if v >= 0 else "#c0392b" for v in dec["remainder"]]

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.10,
    row_heights=[0.55, 0.45],
    specs=[[{"secondary_y": True}], [{"secondary_y": True}]],
    subplot_titles=("Sales, Trend & Seasonal", "Residual & Precipitation"),
)

# ---- top panel: sales + trend (+ seasonal on secondary y) ----
fig.add_trace(
    go.Scatter(
        x=dec["date"],
        y=dec["sales_quantity"],
        name="Sales",
        mode="lines+markers",
        line=dict(color="#2c5282", width=1.2),
        marker=dict(size=4),
        hovertemplate="%{x|%b %d, %Y}<br>Sales: %{y}<extra></extra>",
    ),
    row=1,
    col=1,
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=dec["date"],
        y=dec["trend"],
        name="Trend",
        mode="lines",
        line=dict(color="#e57373", width=2.5),
        hovertemplate="%{x|%b %d, %Y}<br>Trend: %{y:.1f}<extra></extra>",
    ),
    row=1,
    col=1,
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=dec["date"],
        y=dec["seasonal"],
        name="Seasonal",
        mode="lines",
        line=dict(color="#d4a017", width=1, dash="dot"),
        opacity=0.6,
        hovertemplate="%{x|%b %d, %Y}<br>Seasonal: %{y:+.1f}<extra></extra>",
    ),
    row=1,
    col=1,
    secondary_y=True,
)

# ---- bottom panel: residual bars + precipitation bars ----
fig.add_trace(
    go.Bar(
        x=dec["date"],
        y=dec["remainder"],
        name="Residual",
        marker_color=resid_colors,
        hovertemplate="%{x|%b %d, %Y}<br>Residual: %{y:+.1f}<extra></extra>",
    ),
    row=2,
    col=1,
    secondary_y=False,
)

fig.add_trace(
    go.Bar(
        x=dec["date"],
        y=dec["precipitation"],
        name="Precipitation",
        marker_color="#a8d0e6",
        opacity=0.55,
        hovertemplate="%{x|%b %d, %Y}<br>Precip: %{y:.1f} mm<extra></extra>",
    ),
    row=2,
    col=1,
    secondary_y=True,
)

# ---- axes & layout ----
fig.update_yaxes(title_text="Sales / Trend", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Seasonal", row=1, col=1, secondary_y=True, showgrid=False)
fig.update_yaxes(title_text="Residual", row=2, col=1, secondary_y=False)
fig.update_yaxes(
    title_text="Precipitation (mm)", row=2, col=1, secondary_y=True, showgrid=False
)

fig.update_layout(
    height=620,
    template="plotly_white",
    barmode="overlay",
    legend=dict(orientation="h", y=1.08, x=0, bgcolor="rgba(0,0,0,0)"),
    margin=dict(l=60, r=60, t=70, b=40),
)
fig.show()

## Exploration All Shops

In [14]:
CUTOFF = "2024-11-30"
RAIN_BINS = [-0.01, 0.1, 2, 8, 1e9]
RAIN_LABS = ["none", "light", "moderate", "heavy"]
LOW_SALE_PCT = 0.20  # drop shop-days < 20% of that shop's median (data quality)

# ============================================================
# 1. BUILD THE POOLED SHOP-DAY PANEL (all shops)
# ============================================================
print("Building shop-day panel ...")
agg = (
    sellout.groupby(["customer_code", "date"])
    .agg(
        sales_quantity=("sales_quantity", "sum"),
        sales_amount=("sales_amount", "sum"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
    )
    .reset_index()
)

panel = agg.merge(
    weather_df[
        ["date", "latitude", "longitude", "precipitation", "temperature", "windspeed"]
    ],
    on=["date", "latitude", "longitude"],
    how="left",
)

# panel = panel[panel["date"] <= CUTOFF]
panel = panel.dropna(subset=["precipitation", "temperature", "windspeed"])
print(f"  shop-day rows: {len(panel):,}   shops: {panel['customer_code'].nunique()}")

# ============================================================
# 2. SHOP-SPECIFIC DATA QUALITY FILTER
# ============================================================
# Each shop has its own typical volume; the 20% floor is per-shop, not global.
# Also drop rows with zero/negative revenue (returns, refunds) — they break log().
# med = panel.groupby("customer_code")["sales_quantity"].transform("median")
# floor = LOW_SALE_PCT * med
# keep = (
#     (panel["sales_quantity"] >= floor)
#     & (panel["sales_quantity"] > 0)
#     & (panel["sales_amount"] > 0)
# )
# dropped = (~keep).sum()
# print(
#     f"  rows dropped as data-quality outliers: {dropped:,} "
#     f"({dropped/len(panel):.1%})"
# )
# panel = panel[keep].copy()
panel = panel[(panel["sales_quantity"] > 0) & (panel["sales_amount"] > 0)].copy()


# ============================================================
# 3. FEATURES: bands, lags, leads, calendar
# ============================================================
panel = panel.sort_values(["customer_code", "date"]).reset_index(drop=True)
panel["log_q"] = np.log(panel["sales_quantity"])
panel["log_v"] = np.log(panel["sales_amount"])  # revenue outcome
panel["dow"] = panel["date"].dt.dayofweek.astype("category")
panel["month"] = panel["date"].dt.month.astype("category")
panel["trend"] = (panel["date"] - panel["date"].min()).dt.days
panel["band"] = pd.Categorical(
    pd.cut(panel["precipitation"], RAIN_BINS, labels=RAIN_LABS), categories=RAIN_LABS
)

# lag-1 and lead-1 rain bands, computed PER SHOP so the shift doesn't
# leak across shop boundaries.
panel["band_lag1"] = panel.groupby("customer_code")["band"].shift(1)
panel["band_lead1"] = panel.groupby("customer_code")["band"].shift(-1)
panel["gap_lag"] = panel.groupby("customer_code")["date"].diff().dt.days
panel["gap_lead"] = -panel.groupby("customer_code")["date"].diff(-1).dt.days

# only use lag/lead rows where 'yesterday'/'tomorrow' is the actual
# adjacent calendar day — closed-day gaps invalidate the shift
panel["lag_ok"] = panel["gap_lag"] == 1
panel["lead_ok"] = panel["gap_lead"] == 1

irreg_lag = 1 - panel["lag_ok"].mean()
irreg_lead = 1 - panel["lead_ok"].mean()
print(f"  rows where lag-1 is not literal previous day : {irreg_lag:.1%}")
print(f"  rows where lead-1 is not literal next day    : {irreg_lead:.1%}")

# ============================================================
# 4. POOLED MODEL — SALES QUANTITY (primary outcome)
# ============================================================
print("\nFitting pooled model (sales_quantity) ...")
model_df = panel.dropna(subset=["band_lag1", "band_lead1"])
model_df = model_df[model_df["lag_ok"] & model_df["lead_ok"]]
print(f"  rows used in model : {len(model_df):,}")

formula_q = (
    "log_q ~ C(band) + C(band_lag1) + C(band_lead1) "
    "+ temperature + windspeed "
    "+ C(dow) + C(month) + trend + C(customer_code)"
)
mq = smf.ols(formula_q, data=model_df).fit(
    cov_type="cluster", cov_kwds={"groups": model_df["customer_code"]}
)


def report_bands(model, prefix, label):
    print(f"\n  {label}:")
    for term in model.params.index:
        if term.startswith(prefix) and "T." in term:
            band = term.split("T.")[-1].rstrip("]")
            coef, p = model.params[term], model.pvalues[term]
            lo, hi = model.conf_int().loc[term]
            sig = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""
            print(
                f"    {band:<9}: {np.expm1(coef):+6.2%}  "
                f"[{np.expm1(lo):+.2%}, {np.expm1(hi):+.2%}]  "
                f"p={p:.3f} {sig}"
            )


print("\n" + "=" * 60)
print("RESULTS — SALES QUANTITY (log outcome, all shops pooled)")
print("=" * 60)
report_bands(mq, "C(band)[", "Same-day rain (vs dry day)")
report_bands(mq, "C(band_lag1)[", "Yesterday's rain (delay test)")
report_bands(mq, "C(band_lead1)[", "Tomorrow's rain (anticipation test)")
print(
    f"\n  temperature coef: {mq.params['temperature']:+.4f} log-units/°C  "
    f"p={mq.pvalues['temperature']:.3f}"
)
print(
    f"  windspeed   coef: {mq.params['windspeed']:+.4f}             "
    f"p={mq.pvalues['windspeed']:.3f}"
)

# ============================================================
# 5. SAME MODEL — SALES AMOUNT (revenue)
# ============================================================
print("\nFitting pooled model (sales_amount) ...")
formula_v = formula_q.replace("log_q", "log_v")
mv = smf.ols(formula_v, data=model_df).fit(
    cov_type="cluster", cov_kwds={"groups": model_df["customer_code"]}
)

print("\n" + "=" * 60)
print("RESULTS — SALES AMOUNT (revenue)")
print("=" * 60)
report_bands(mv, "C(band)[", "Same-day rain (vs dry day)")
report_bands(mv, "C(band_lag1)[", "Yesterday's rain (delay test)")
report_bands(mv, "C(band_lead1)[", "Tomorrow's rain (anticipation test)")

Building shop-day panel ...
  shop-day rows: 427,877   shops: 608
  rows where lag-1 is not literal previous day : 10.2%
  rows where lead-1 is not literal next day    : 10.2%

Fitting pooled model (sales_quantity) ...
  rows used in model : 331,997

RESULTS — SALES QUANTITY (log outcome, all shops pooled)

  Same-day rain (vs dry day):
    light    : -4.17%  [-4.60%, -3.73%]  p=0.000 ***
    moderate : -0.47%  [-0.92%, -0.02%]  p=0.042 **
    heavy    : -10.20%  [-10.82%, -9.58%]  p=0.000 ***

  Yesterday's rain (delay test):
    light    : -1.78%  [-2.15%, -1.41%]  p=0.000 ***
    moderate : +1.63%  [+1.21%, +2.06%]  p=0.000 ***
    heavy    : -4.64%  [-5.24%, -4.04%]  p=0.000 ***

  Tomorrow's rain (anticipation test):
    light    : -4.45%  [-4.96%, -3.94%]  p=0.000 ***
    moderate : +0.22%  [-0.20%, +0.64%]  p=0.312 
    heavy    : -6.92%  [-7.50%, -6.34%]  p=0.000 ***

  temperature coef: +0.0149 log-units/°C  p=0.000
  windspeed   coef: -0.0003             p=0.051

Fitting pool

In [17]:
def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else "ns"


def pull_qty(band):
    term = f"C(band)[T.{band}]"
    coef = mq.params[term]
    lo, hi = mq.conf_int().loc[term]
    p = mq.pvalues[term]
    return np.expm1(coef) * 100, np.expm1(lo) * 100, np.expm1(hi) * 100, p


bands = ["light", "moderate", "heavy"]
vals, err_lo, err_hi, pvals = [], [], [], []
for b in bands:
    v, lo, hi, p = pull_qty(b)
    vals.append(v)
    err_lo.append(v - lo)
    err_hi.append(hi - v)
    pvals.append(p)

text_labels = [f"{v:+.2f}%  {stars(p)}" for v, p in zip(vals, pvals)]
colors = ["#a8d0e6", "#5b9bd5", "#1f4e79"]

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=bands,
        y=vals,
        marker_color=colors,
        error_y=dict(
            type="data",
            symmetric=False,
            array=err_hi,
            arrayminus=err_lo,
            color="#333",
            thickness=1.5,
            width=8,
        ),
        text=text_labels,
        textposition="outside",
        hovertemplate="<b>%{x} rain</b><br>%{y:+.2f}% vs dry day<extra></extra>",
    )
)

fig.add_hline(y=0, line_color="gray", line_width=1)

fig.update_layout(
    title=dict(
        text="<b>Same-day effect of rain on cigarette quantity sold — Paris, All shops</b>"
        "<br><sub>% change in sales_quantity vs a comparable dry day</sub>",
        x=0.02,
        xanchor="left",
    ),
    yaxis_title="% change in quantity vs dry day",
    xaxis_title="Rainfall band  (light: 0.1–2mm  ·  moderate: 2–8mm  ·  heavy: >8mm)",
    template="plotly_white",
    showlegend=False,
    height=460,
    margin=dict(l=70, r=30, t=90, b=70),
)
fig.show()

In [18]:
RAIN_MM = 3
RAIN_BINS = [-0.01, 0.1, 2, 8, 1e9]
RAIN_LABS = ["none", "light", "moderate", "heavy"]
MIN_DAYS = 30

shop_results = []
skipped = 0
cur_shop = 1

# Get common customer codes between sellout and sellin
common_shops = set(sellout["customer_code"].unique()) & set(
    sellin["customer_code"].unique()
)

# Keep only common shops
shops = list(common_shops)
print(f"Running STL for {len(shops)} shops ...")

for shop in shops:
    print("Shop # %d / %d" % (cur_shop, len(shops)), end="\r")
    cur_shop += 1
    # ── Prepare shop data (same as single-shop) ───────────────────────────────
    sp = sellout[sellout["customer_code"] == shop].copy()
    sp.drop_duplicates(inplace=True)
    sp = (
        sp.groupby("date")
        .agg(
            sales_quantity=("sales_quantity", "sum"),
            sales_amount=("sales_amount", "sum"),
            latitude=("latitude", "first"),
            longitude=("longitude", "first"),
        )
        .reset_index()
    )
    sp = sp.merge(weather_df, on=["date", "latitude", "longitude"], how="left")

    if sp["precipitation"].isna().all():  # no weather data at all for this shop
        skipped += 1
        continue

    sp = sp.dropna(subset=["precipitation"])

    if len(sp) < MIN_DAYS:
        skipped += 1
        continue

    sp["rained"] = sp["precipitation"] > RAIN_MM

    # ── STL ───────────────────────────────────────────────────────────────────
    s = sp.set_index("date")["sales_quantity"].asfreq("D")
    s = s.interpolate("time", limit_direction="both")

    try:
        stl = sm.tsa.STL(s, period=7, robust=True).fit()
    except Exception:
        skipped += 1
        continue

    dec = pd.DataFrame(
        {
            "trend": stl.trend,
            "seasonal": stl.seasonal,
            "remainder": stl.resid,
        }
    )
    dec.index.name = "date"
    dec = dec.reset_index()
    dec = dec.merge(
        sp[["date", "precipitation", "rained", "sales_quantity"]],
        on="date",
        how="inner",
    )

    # Keep only real selling days
    real_days = sp.loc[sp["sales_quantity"] > 0, "date"]
    dec = dec[dec["date"].isin(real_days)]

    if len(dec) < MIN_DAYS:
        skipped += 1
        continue

    # ── (1) Correlation & t-test ──────────────────────────────────────────────
    r = dec[["remainder", "precipitation"]].corr().iloc[0, 1]
    t_stat, p_ttest = stats.ttest_ind(
        dec.loc[~dec["rained"], "remainder"],
        dec.loc[dec["rained"], "remainder"],
        equal_var=False,
    )
    dry_mean = dec.loc[~dec["rained"], "remainder"].mean()
    rainy_mean = dec.loc[dec["rained"], "remainder"].mean()

    # ── (2) Band means ────────────────────────────────────────────────────────
    dec["band"] = pd.cut(dec["precipitation"], RAIN_BINS, labels=RAIN_LABS)
    band_means = (
        dec.groupby("band", observed=True)["remainder"].mean().reindex(RAIN_LABS)
    )

    # ── (3) ANOVA ─────────────────────────────────────────────────────────────
    groups = [g["remainder"].values for _, g in dec.groupby("band", observed=True)]
    F, p_anova = stats.f_oneway(*groups) if len(groups) >= 2 else (np.nan, np.nan)

    shop_results.append(
        {
            "customer_code": shop,
            "n_days": len(dec),
            "corr": r,
            "dry_mean": dry_mean,
            "rainy_mean": rainy_mean,
            "diff": rainy_mean - dry_mean,
            "p_ttest": p_ttest,
            "mean_none": band_means["none"],
            "mean_light": band_means["light"],
            "mean_moderate": band_means["moderate"],
            "mean_heavy": band_means["heavy"],
            "anova_F": F,
            "anova_p": p_anova,
        }
    )

results_df = pd.DataFrame(shop_results)
print(f"Shops completed: {len(results_df)}  |  skipped: {skipped}")
results_df.head()

Running STL for 613 shops ...
Shops completed: 606  |  skipped: 7


,customer_code,n_days,corr,dry_mean,rainy_mean,diff,p_ttest,mean_none,mean_light,mean_moderate,mean_heavy,anova_F,anova_p
0,0011t000011b1PXAAY,808,-0.038495,1.499396,0.779764,-0.719633,0.415463,1.819943,0.953567,0.831348,0.938346,0.418729,0.739610
1,0011t000011b1QvAAI,633,-0.055843,0.878373,-0.819479,-1.697852,0.140020,1.539940,0.671592,-0.587028,-2.491511,2.247770,0.081640
2,0011t000011b20vAAA,586,0.006918,1.197708,1.224585,0.026877,0.982558,1.309586,0.657952,1.650492,1.202921,0.116640,0.950343
3,0011t000011b9RbAAI,668,-0.050050,1.037007,0.633259,-0.403748,0.793104,1.090013,1.573472,0.699158,-0.814258,0.338329,0.797629
4,0011t000011b1iIAAQ,660,0.007968,-0.974672,-0.479225,0.495448,0.841291,-1.561778,-0.525115,1.445995,-3.267779,0.799265,0.494520


In [19]:
print("=" * 55)
print("AGGREGATED STL RESULTS ACROSS ALL SHOPS")
print("=" * 55)

print(f"\n  Shops analysed          : {len(results_df)}")
print(f"  Median correlation      : {results_df['corr'].median():+.3f}")
print(
    f"  Shops with corr < 0     : {(results_df['corr'] < 0).sum()} ({(results_df['corr'] < 0).mean():.1%})"
)

print(f"\n  Avg remainder — dry     : {results_df['dry_mean'].mean():+.2f}")
print(f"  Avg remainder — rainy   : {results_df['rainy_mean'].mean():+.2f}")
print(f"  Avg difference          : {results_df['diff'].mean():+.2f}")
print(
    f"  Shops sig. (p<0.05)     : {(results_df['p_ttest'] < 0.05).sum()} ({(results_df['p_ttest'] < 0.05).mean():.1%})"
)

print("\n  Avg STL remainder by band (averaged across shops):")
for band in RAIN_LABS:
    col = f"mean_{band}"
    print(
        f"    {band:<10}: {results_df[col].mean():+.2f}  (median: {results_df[col].median():+.2f})"
    )

sig_anova = (results_df["anova_p"] < 0.05).sum()
print(
    f"\n  Shops with sig. ANOVA (p<0.05): {sig_anova} ({sig_anova/len(results_df):.1%})"
)

AGGREGATED STL RESULTS ACROSS ALL SHOPS

  Shops analysed          : 606
  Median correlation      : -0.029
  Shops with corr < 0     : 478 (78.9%)

  Avg remainder — dry     : +1.03
  Avg remainder — rainy   : +0.35
  Avg difference          : -0.67
  Shops sig. (p<0.05)     : 72 (11.9%)

  Avg STL remainder by band (averaged across shops):
    none      : +1.32  (median: +1.22)
    light     : +0.75  (median: +0.68)
    moderate  : +0.47  (median: +0.51)
    heavy     : -0.14  (median: -0.09)

  Shops with sig. ANOVA (p<0.05): 58 (9.6%)


In [20]:
import plotly.graph_objects as go
from scipy import stats as scipy_stats

bands = ["light", "moderate", "heavy"]
cols = ["mean_light", "mean_moderate", "mean_heavy"]

means = [results_df[c].mean() for c in cols]
sems = [results_df[c].sem() for c in cols]  # standard error across shops
ns = len(results_df)

# 95% CI using t-distribution
t_crit = scipy_stats.t.ppf(0.975, df=ns - 1)
ci = [t_crit * se for se in sems]

# p-value: is the cross-shop mean significantly different from 0?
pvals = [scipy_stats.ttest_1samp(results_df[c].dropna(), 0).pvalue for c in cols]


def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else "ns"


text_labels = [f"{v:+.2f}  {stars(p)}" for v, p in zip(means, pvals)]
colors = ["#a8d0e6", "#5b9bd5", "#1f4e79"]

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=bands,
        y=means,
        marker_color=colors,
        error_y=dict(
            type="data",
            symmetric=True,
            array=ci,
            color="#333",
            thickness=1.5,
            width=8,
        ),
        text=text_labels,
        textposition="outside",
        hovertemplate="<b>%{x} rain</b><br>avg remainder: %{y:+.2f}<extra></extra>",
    )
)

fig.add_hline(y=0, line_color="gray", line_width=1)

fig.update_layout(
    title=dict(
        text=f"<b>Effect of rain on STL remainder — {ns} shops</b>"
        "<br><sub>Mean STL remainder by rainfall band, averaged across shops (error bars = 95% CI)</sub>",
        x=0.02,
        xanchor="left",
    ),
    yaxis_title="Mean STL remainder (sales units above/below trend)",
    xaxis_title="Rainfall band  (light: 0.1–2mm  ·  moderate: 2–8mm  ·  heavy: >8mm)",
    template="plotly_white",
    showlegend=False,
    height=460,
    margin=dict(l=70, r=30, t=90, b=70),
)
fig.show()

## Exploration of New Regression Method

### Single Shop

In [21]:
selected_customer_data = sellout[
    sellout["customer_code"] == "0011t000011b2hEAAQ"
].copy()
selected_customer_data.drop_duplicates(inplace=True)
new_data = (
    selected_customer_data.groupby("date")
    .agg(
        sales_quantity=("sales_quantity", "sum"),
        sales_amount=("sales_amount", "sum"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
    )
    .reset_index()
)

# ── Merge weather ─────────────────────────────────────────────────────────────
merged_data = new_data.merge(
    weather_df[
        ["date", "latitude", "longitude", "precipitation", "temperature", "windspeed"]
    ],
    on=["date", "latitude", "longitude"],
    how="left",
).dropna(subset=["precipitation", "temperature", "windspeed"])

print(
    f"Shop rows: {len(merged_data):,}  |  date range: {merged_data['date'].min().date()} → {merged_data['date'].max().date()}"
)

# ── Quality filter ────────────────────────────────────────────────────────────
RAIN_BINS = [-0.01, 0.1, 2, 8, 1e9]
RAIN_LABS = ["none", "light", "moderate", "heavy"]
LOW_SALE_PCT = 0.20

m = merged_data.copy()
# median_q = m["sales_quantity"].median()
# m = m[
#     (m["sales_quantity"] >= LOW_SALE_PCT * median_q) & (m["sales_quantity"] > 0)
# ].copy()
m = m[m["sales_quantity"] > 0].copy()  # only drop true zeros (breaks log)


# ── Features ──────────────────────────────────────────────────────────────────
m = m.sort_values("date").reset_index(drop=True)
m["log_q"] = np.log(m["sales_quantity"])
m["dow"] = m["date"].dt.dayofweek.astype("category")
m["month"] = m["date"].dt.month.astype("category")
m["trend"] = (m["date"] - m["date"].min()).dt.days
m["band"] = pd.Categorical(
    pd.cut(m["precipitation"], RAIN_BINS, labels=RAIN_LABS), categories=RAIN_LABS
)
m["band_lag1"] = m["band"].shift(1)
m["band_lead1"] = m["band"].shift(-1)
m["gap_lag"] = m["date"].diff().dt.days
m["gap_lead"] = -m["date"].diff(-1).dt.days
m["lag_ok"] = m["gap_lag"] == 1
m["lead_ok"] = m["gap_lead"] == 1

# ── Fit model ─────────────────────────────────────────────────────────────────
model_df = m.dropna(subset=["band_lag1", "band_lead1"])
model_df = model_df[model_df["lag_ok"] & model_df["lead_ok"]]
print(f"Rows used in model: {len(model_df):,}")

formula = (
    "log_q ~ C(band) + C(band_lag1) + C(band_lead1)"
    " + temperature + windspeed"
    " + C(dow) + C(month) + trend"
)
model = smf.ols(formula, data=model_df).fit(cov_type="HC3")


# ── Results ───────────────────────────────────────────────────────────────────
def report_bands(model, prefix, label):
    print(f"\n  {label}:")
    for term in model.params.index:
        if term.startswith(prefix) and "T." in term:
            band = term.split("T.")[-1].rstrip("]")
            coef, p = model.params[term], model.pvalues[term]
            lo, hi = model.conf_int().loc[term]
            sig = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else ""
            print(
                f"    {band:<9}: {np.expm1(coef):+6.2%}"
                f"  [{np.expm1(lo):+.2%}, {np.expm1(hi):+.2%}]"
                f"  p={p:.3f} {sig}"
            )


print("\n" + "=" * 60)
print("RESULTS — shop 0011t000011b5TiAAI — SALES QUANTITY")
print("=" * 60)
report_bands(model, "C(band)[", "Same-day rain (vs dry day)")
report_bands(model, "C(band_lag1)[", "Yesterday's rain (delay test)")
report_bands(model, "C(band_lead1)[", "Tomorrow's rain (anticipation test)")
print(
    f"\n  temperature: {model.params['temperature']:+.4f}  p={model.pvalues['temperature']:.3f}"
)
print(
    f"  windspeed  : {model.params['windspeed']:+.4f}  p={model.pvalues['windspeed']:.3f}"
)
print(f"\n  R²={model.rsquared:.3f}   n={len(model_df)}")

Shop rows: 813  |  date range: 2024-01-02 → 2026-03-31
Rows used in model: 800

RESULTS — shop 0011t000011b5TiAAI — SALES QUANTITY

  Same-day rain (vs dry day):
    light    : -5.37%  [-12.38%, +2.20%]  p=0.160 
    moderate : -0.30%  [-8.59%, +8.74%]  p=0.945 
    heavy    : -5.55%  [-15.96%, +6.14%]  p=0.337 

  Yesterday's rain (delay test):
    light    : +2.15%  [-4.44%, +9.19%]  p=0.532 
    moderate : +3.43%  [-4.57%, +12.11%]  p=0.411 
    heavy    : +2.93%  [-7.83%, +14.94%]  p=0.608 

  Tomorrow's rain (anticipation test):
    light    : -5.86%  [-12.22%, +0.96%]  p=0.091 *
    moderate : -1.25%  [-8.11%, +6.13%]  p=0.733 
    heavy    : -7.21%  [-16.77%, +3.45%]  p=0.177 

  temperature: +0.0123  p=0.001
  windspeed  : -0.0022  p=0.374

  R²=0.516   n=800


In [22]:
def stars(p):
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else "ns"


bands = ["light", "moderate", "heavy"]
vals, err_lo, err_hi, pvals = [], [], [], []

for band in bands:
    term = f"C(band)[T.{band}]"
    coef = model.params[term]
    lo, hi = model.conf_int().loc[term]
    p = model.pvalues[term]
    v = np.expm1(coef) * 100
    vals.append(v)
    err_lo.append(v - np.expm1(lo) * 100)
    err_hi.append(np.expm1(hi) * 100 - v)
    pvals.append(p)

text_labels = [f"{v:+.2f}%  {stars(p)}" for v, p in zip(vals, pvals)]
colors = ["#a8d0e6", "#5b9bd5", "#1f4e79"]

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=bands,
        y=vals,
        marker_color=colors,
        error_y=dict(
            type="data",
            symmetric=False,
            array=err_hi,
            arrayminus=err_lo,
            color="#333",
            thickness=1.5,
            width=8,
        ),
        text=text_labels,
        textposition="outside",
        hovertemplate="<b>%{x} rain</b><br>%{y:+.2f}% vs dry day<extra></extra>",
    )
)

fig.add_hline(y=0, line_color="gray", line_width=1)

fig.update_layout(
    title=dict(
        text="<b>Same-day effect of rain on cigarette quantity — shop 0011t000011b5TiAAI</b>"
        "<br><sub>% change in sales_quantity vs a comparable dry day</sub>",
        x=0.02,
        xanchor="left",
    ),
    yaxis_title="% change in quantity vs dry day",
    xaxis_title="Rainfall band  (light: 0.1–2mm  ·  moderate: 2–8mm  ·  heavy: >8mm)",
    template="plotly_white",
    showlegend=False,
    height=460,
    margin=dict(l=70, r=30, t=90, b=70),
)
fig.show()

In [23]:
import calendar

months = [str(i) for i in range(2, 13)]  # 2-12, January is reference
vals, err_lo, err_hi, pvals = [], [], [], []

for mo in months:
    term = f"C(month)[T.{mo}]"
    coef = model.params[term]
    lo, hi = model.conf_int().loc[term]
    p = model.pvalues[term]
    v = np.expm1(coef) * 100
    vals.append(v)
    err_lo.append(v - np.expm1(lo) * 100)
    err_hi.append(np.expm1(hi) * 100 - v)
    pvals.append(p)

month_names = [calendar.month_abbr[int(m)] for m in months]
text_labels = [f"{v:+.1f}%  {stars(p)}" for v, p in zip(vals, pvals)]
colors = ["#c7395f" if v < 0 else "#2a9d8f" for v in vals]

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=month_names,
        y=vals,
        marker_color=colors,
        error_y=dict(
            type="data",
            symmetric=False,
            array=err_hi,
            arrayminus=err_lo,
            color="#333",
            thickness=1.5,
            width=8,
        ),
        text=text_labels,
        textposition="outside",
        hovertemplate="<b>%{x}</b><br>%{y:+.2f}% vs January<extra></extra>",
    )
)
fig.add_hline(y=0, line_color="gray", line_width=1)

fig.update_layout(
    title=dict(
        text="<b>Monthly seasonality — sales vs January</b>"
        "<br><sub>% change in sales_quantity, holding weather & day-of-week constant</sub>",
        x=0.02,
        xanchor="left",
    ),
    yaxis_title="% change in quantity vs January",
    xaxis_title="Month",
    template="plotly_white",
    showlegend=False,
    height=460,
    margin=dict(l=70, r=30, t=90, b=70),
)
fig.show()

In [24]:
dow_nums = ["1", "2", "3", "4", "5", "6"]  # Monday=0 is reference
dow_names = ["Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
vals, err_lo, err_hi, pvals = [], [], [], []

for d in dow_nums:
    term = f"C(dow)[T.{d}]"
    coef = model.params[term]
    lo, hi = model.conf_int().loc[term]
    p = model.pvalues[term]
    v = np.expm1(coef) * 100
    vals.append(v)
    err_lo.append(v - np.expm1(lo) * 100)
    err_hi.append(np.expm1(hi) * 100 - v)
    pvals.append(p)

text_labels = [f"{v:+.1f}%  {stars(p)}" for v, p in zip(vals, pvals)]
colors = ["#c7395f" if v < 0 else "#2a9d8f" for v in vals]

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=dow_names,
        y=vals,
        marker_color=colors,
        error_y=dict(
            type="data",
            symmetric=False,
            array=err_hi,
            arrayminus=err_lo,
            color="#333",
            thickness=1.5,
            width=8,
        ),
        text=text_labels,
        textposition="outside",
        hovertemplate="<b>%{x}</b><br>%{y:+.2f}% vs Monday<extra></extra>",
    )
)
fig.add_hline(y=0, line_color="gray", line_width=1)

fig.update_layout(
    title=dict(
        text="<b>Day-of-week effect — sales vs Monday</b>"
        "<br><sub>% change in sales_quantity, holding weather & month constant</sub>",
        x=0.02,
        xanchor="left",
    ),
    yaxis_title="% change in quantity vs Monday",
    xaxis_title="Day of week",
    template="plotly_white",
    showlegend=False,
    height=460,
    margin=dict(l=70, r=30, t=90, b=70),
)
fig.show()

In [25]:
temp_min, temp_max = m["temperature"].min(), m["temperature"].max()
temp_grid = np.linspace(temp_min, temp_max, 50)
temp_ref = m["temperature"].median()  # baseline

coef = model.params["temperature"]
se = model.bse["temperature"]

# Effect at each temp relative to median temp
log_effect = coef * (temp_grid - temp_ref)
pct_effect = (np.expm1(log_effect)) * 100
pct_lo = (np.expm1(log_effect - 1.96 * se * (temp_grid - temp_ref))) * 100
pct_hi = (np.expm1(log_effect + 1.96 * se * (temp_grid - temp_ref))) * 100

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=temp_grid,
        y=pct_hi,
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip",
    )
)
fig.add_trace(
    go.Scatter(
        x=temp_grid,
        y=pct_lo,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor="rgba(31,78,121,0.15)",
        showlegend=False,
        hoverinfo="skip",
    )
)
fig.add_trace(
    go.Scatter(
        x=temp_grid,
        y=pct_effect,
        mode="lines",
        line=dict(color="#1f4e79", width=2.5),
        hovertemplate="Temp: %{x:.1f}°C<br>Effect: %{y:+.2f}%<extra></extra>",
        name="Effect",
    )
)
fig.add_hline(y=0, line_color="gray", line_width=1)
fig.add_vline(
    x=temp_ref,
    line_color="gray",
    line_width=1,
    line_dash="dot",
    annotation_text=f"median {temp_ref:.1f}°C",
    annotation_position="top",
)

fig.update_layout(
    title=dict(
        text="<b>Temperature effect across observed range</b>"
        f"<br><sub>% change in sales_quantity vs median temperature ({temp_ref:.1f}°C)</sub>",
        x=0.02,
        xanchor="left",
    ),
    yaxis_title="% change vs median-temp day",
    xaxis_title="Temperature (°C)",
    template="plotly_white",
    height=460,
    margin=dict(l=70, r=30, t=90, b=70),
)
fig.show()

### All shops

In [26]:
rain_shop_results = []
rain_skipped = 0
rain_cur_shop = 1

# Get common customer codes between sellout and sellin
rain_common_shops = set(sellout["customer_code"].unique()) & set(
    sellin["customer_code"].unique()
)

# Keep only common shops
rain_shops = list(rain_common_shops)
print(f"Running rain-band OLS for {len(rain_shops)} shops ...")

RAIN_BINS = [-0.01, 0.1, 2, 8, 1e9]
RAIN_LABS = ["none", "light", "moderate", "heavy"]
RAIN_MIN_ROWS = 30  # need enough rows to fit the model

for rain_shop in rain_shops:
    print("Shop # %d / %d" % (rain_cur_shop, len(rain_shops)), end="\r")
    rain_cur_shop += 1

    # ── Prepare shop data (same as single-shop) ───────────────────────────────
    rain_sp = sellout[sellout["customer_code"] == rain_shop].copy()
    rain_sp.drop_duplicates(inplace=True)

    rain_new_data = (
        rain_sp.groupby("date")
        .agg(
            sales_quantity=("sales_quantity", "sum"),
            sales_amount=("sales_amount", "sum"),
            latitude=("latitude", "first"),
            longitude=("longitude", "first"),
        )
        .reset_index()
    )

    # ── Merge weather ─────────────────────────────────────────────────────────
    rain_merged = rain_new_data.merge(
        weather_df[
            [
                "date",
                "latitude",
                "longitude",
                "precipitation",
                "temperature",
                "windspeed",
            ]
        ],
        on=["date", "latitude", "longitude"],
        how="left",
    ).dropna(subset=["precipitation", "temperature", "windspeed"])

    # ── Quality filter ────────────────────────────────────────────────────────
    rain_m = rain_merged[rain_merged["sales_quantity"] > 0].copy()

    if len(rain_m) < RAIN_MIN_ROWS:
        rain_skipped += 1
        continue

    # ── Features (same-day rain only) ─────────────────────────────────────────
    rain_m = rain_m.sort_values("date").reset_index(drop=True)
    rain_m["log_q"] = np.log(rain_m["sales_quantity"])
    rain_m["dow"] = rain_m["date"].dt.dayofweek.astype("category")
    rain_m["month"] = rain_m["date"].dt.month.astype("category")
    rain_m["trend"] = (rain_m["date"] - rain_m["date"].min()).dt.days
    rain_m["band"] = pd.Categorical(
        pd.cut(rain_m["precipitation"], RAIN_BINS, labels=RAIN_LABS),
        categories=RAIN_LABS,
    )

    # Need at least 2 rain bands present (including the "none" baseline)
    if rain_m["band"].nunique(dropna=True) < 2:
        rain_skipped += 1
        continue

    # ── Fit model (same-day band only) ────────────────────────────────────────
    rain_formula = (
        "log_q ~ C(band)" " + temperature + windspeed" " + C(dow) + C(month) + trend"
    )

    try:
        rain_model = smf.ols(rain_formula, data=rain_m).fit(cov_type="HC3")
    except Exception:
        rain_skipped += 1
        continue

    # ── Collect same-day rain band coefficients ───────────────────────────────
    for rain_term in rain_model.params.index:
        if rain_term.startswith("C(band)[") and "T." in rain_term:
            rain_band = rain_term.split("T.")[-1].rstrip("]")
            rain_coef = rain_model.params[rain_term]
            rain_p = rain_model.pvalues[rain_term]
            rain_lo, rain_hi = rain_model.conf_int().loc[rain_term]

            rain_shop_results.append(
                {
                    "customer_code": rain_shop,
                    "band": rain_band,
                    "effect_pct": np.expm1(rain_coef),
                    "ci_low_pct": np.expm1(rain_lo),
                    "ci_high_pct": np.expm1(rain_hi),
                    "coef": rain_coef,
                    "p_value": rain_p,
                    "n_rows": len(rain_m),
                    "r_squared": rain_model.rsquared,
                }
            )

print(f"\nDone. Fitted: {len(rain_shops) - rain_skipped}  |  Skipped: {rain_skipped}")

rain_results_df = pd.DataFrame(rain_shop_results)
rain_results_df.head()

Running rain-band OLS for 613 shops ...


/home/asad/anaconda3/envs/footfall_explorer/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:2014: RuntimeWarning: divide by zero encountered in divide
  self.het_scale = (self.wresid / (1 - h))**2


Shop # 613 / 613
Done. Fitted: 606  |  Skipped: 7


,customer_code,band,effect_pct,ci_low_pct,ci_high_pct,coef,p_value,n_rows,r_squared
0,0011t000011b1PXAAY,light,-0.037453,-0.144995,0.083616,-0.038172,0.527722,808,0.364901
1,0011t000011b1PXAAY,moderate,0.077354,-0.049052,0.220564,0.074508,0.241961,808,0.364901
2,0011t000011b1PXAAY,heavy,-0.022955,-0.156791,0.132124,-0.023222,0.757355,808,0.364901
3,0011t000011b1QvAAI,light,-0.136467,-0.240177,-0.018600,-0.146723,0.024604,633,0.467870
4,0011t000011b1QvAAI,moderate,-0.100283,-0.207149,0.020986,-0.105675,0.101416,633,0.467870


In [27]:
print("=" * 55)
print("AGGREGATED OLS RAIN RESULTS ACROSS ALL SHOPS")
print("=" * 55)

n_shops_fit = rain_results_df["customer_code"].nunique()
print(f"\n  Shops analysed              : {n_shops_fit}")
print(f"  Shops skipped               : {rain_skipped}")
print(
    f"  Median R²                   : {rain_results_df.groupby('customer_code')['r_squared'].first().median():.3f}"
)

print("\n  Avg sales effect by band (vs dry day, averaged across shops):")
for rain_b in ["light", "moderate", "heavy"]:
    rain_sub = rain_results_df[rain_results_df["band"] == rain_b]
    if len(rain_sub) == 0:
        print(f"    {rain_b:<10}: no shops")
        continue
    rain_mean = rain_sub["effect_pct"].mean()
    rain_median = rain_sub["effect_pct"].median()
    rain_neg = (rain_sub["effect_pct"] < 0).sum()
    rain_sig = (rain_sub["p_value"] < 0.05).sum()
    print(
        f"    {rain_b:<10}: {rain_mean:+6.2%}  (median: {rain_median:+.2%})"
        f"  |  shops negative: {rain_neg}/{len(rain_sub)} ({rain_neg/len(rain_sub):.1%})"
        f"  |  shops sig. (p<0.05): {rain_sig} ({rain_sig/len(rain_sub):.1%})"
    )

print(f"\n  Total shop-band rows        : {len(rain_results_df)}")

AGGREGATED OLS RAIN RESULTS ACROSS ALL SHOPS

  Shops analysed              : 606
  Shops skipped               : 7
  Median R²                   : 0.407

  Avg sales effect by band (vs dry day, averaged across shops):
    light     : -4.45%  (median: -4.97%)  |  shops negative: 483/606 (79.7%)  |  shops sig. (p<0.05): 84 (13.9%)
    moderate  : -2.36%  (median: -2.50%)  |  shops negative: 427/606 (70.5%)  |  shops sig. (p<0.05): 22 (3.6%)
    heavy     : -9.21%  (median: -9.54%)  |  shops negative: 572/606 (94.4%)  |  shops sig. (p<0.05): 133 (21.9%)

  Total shop-band rows        : 1818


In [28]:
rain_bands_plot = ["light", "moderate", "heavy"]

# Per-band stats across shops
rain_means = []
rain_sems = []
rain_pvals = []
rain_ns_per_band = []

for rain_b in rain_bands_plot:
    rain_vals = rain_results_df.loc[
        rain_results_df["band"] == rain_b, "effect_pct"
    ].dropna()
    rain_means.append(rain_vals.mean())
    rain_sems.append(rain_vals.sem())
    rain_ns_per_band.append(len(rain_vals))
    rain_pvals.append(
        scipy_stats.ttest_1samp(rain_vals, 0).pvalue
        if len(rain_vals) > 1
        else float("nan")
    )

# 95% CI using t-distribution (per band, since n can differ)
rain_ci = [
    scipy_stats.t.ppf(0.975, df=n - 1) * se if n > 1 else 0
    for se, n in zip(rain_sems, rain_ns_per_band)
]


def rain_stars(p):
    if p != p:  # NaN check
        return ""
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else "ns"


rain_text_labels = [
    f"{v:+.1%}  {rain_stars(p)}" for v, p in zip(rain_means, rain_pvals)
]
rain_colors = ["#a8d0e6", "#5b9bd5", "#1f4e79"]

rain_fig = go.Figure()
rain_fig.add_trace(
    go.Bar(
        x=rain_bands_plot,
        y=rain_means,
        marker_color=rain_colors,
        error_y=dict(
            type="data",
            symmetric=True,
            array=rain_ci,
            color="#333",
            thickness=1.5,
            width=8,
        ),
        text=rain_text_labels,
        textposition="outside",
        hovertemplate="<b>%{x} rain</b><br>avg effect: %{y:+.2%}<extra></extra>",
    )
)

rain_fig.add_hline(y=0, line_color="gray", line_width=1)

rain_fig.update_layout(
    title=dict(
        text=f"<b>Effect of same-day rain on sales — {n_shops_fit} shops</b>"
        "<br><sub>Mean % effect on sales by rainfall band vs dry days, averaged across shops (error bars = 95% CI)</sub>",
        x=0.02,
        xanchor="left",
    ),
    yaxis_title="Mean sales effect vs dry day",
    yaxis_tickformat=".0%",
    xaxis_title="Rainfall band  (light: 0.1–2mm  ·  moderate: 2–8mm  ·  heavy: >8mm)",
    template="plotly_white",
    showlegend=False,
    height=460,
    margin=dict(l=70, r=30, t=90, b=70),
)
rain_fig.show()

In [29]:
print(rain_results_df.shape)
rain_results_df.head()

(1818, 9)


,customer_code,band,effect_pct,ci_low_pct,ci_high_pct,coef,p_value,n_rows,r_squared
0,0011t000011b1PXAAY,light,-0.037453,-0.144995,0.083616,-0.038172,0.527722,808,0.364901
1,0011t000011b1PXAAY,moderate,0.077354,-0.049052,0.220564,0.074508,0.241961,808,0.364901
2,0011t000011b1PXAAY,heavy,-0.022955,-0.156791,0.132124,-0.023222,0.757355,808,0.364901
3,0011t000011b1QvAAI,light,-0.136467,-0.240177,-0.018600,-0.146723,0.024604,633,0.467870
4,0011t000011b1QvAAI,moderate,-0.100283,-0.207149,0.020986,-0.105675,0.101416,633,0.467870


In [108]:
# ── Setup ─────────────────────────────────────────────────────────────────────
shop_results = []
skipped = 0
cur_shop = 1

common_shops = set(sellout["customer_code"].unique()) & set(
    sellin["customer_code"].unique()
)
shops = list(common_shops)
print(f"Running OLS for {len(shops)} shops ...")

RAIN_BINS = [-0.01, 0.1, 2, 8, 1e9]
RAIN_LABS = ["none", "light", "moderate", "heavy"]
MIN_ROWS = 30

# ── Per-shop loop ─────────────────────────────────────────────────────────────
for shop in shops:
    print(f"Shop # {cur_shop} / {len(shops)}", end="\r")
    cur_shop += 1

    # Prepare shop data
    sp = sellout[sellout["customer_code"] == shop].copy()
    sp.drop_duplicates(inplace=True)

    new_data = (
        sp.groupby("date")
        .agg(
            sales_quantity=("sales_quantity", "sum"),
            sales_amount=("sales_amount", "sum"),
            latitude=("latitude", "first"),
            longitude=("longitude", "first"),
        )
        .reset_index()
    )

    # Merge weather
    merged = new_data.merge(
        weather_df[
            [
                "date",
                "latitude",
                "longitude",
                "precipitation",
                "temperature",
                "windspeed",
            ]
        ],
        on=["date", "latitude", "longitude"],
        how="left",
    ).dropna(subset=["precipitation", "temperature", "windspeed"])

    # Quality filter
    m = merged[merged["sales_quantity"] > 0].copy()
    if len(m) < MIN_ROWS:
        skipped += 1
        continue

    # Features
    m = m.sort_values("date").reset_index(drop=True)
    m["log_q"] = np.log(m["sales_quantity"])
    m["dow"] = m["date"].dt.dayofweek.astype("category")
    m["month"] = m["date"].dt.month.astype("category")
    m["trend"] = (m["date"] - m["date"].min()).dt.days
    m["band"] = pd.Categorical(
        pd.cut(m["precipitation"], RAIN_BINS, labels=RAIN_LABS),
        categories=RAIN_LABS,
    )

    # Need at least 2 rain bands present
    if m["band"].nunique(dropna=True) < 2:
        skipped += 1
        continue

    # Fit model
    formula = (
        "log_q ~ C(band)" " + temperature + windspeed" " + C(dow) + C(month) + trend"
    )
    try:
        model = smf.ols(formula, data=m).fit(cov_type="HC3")
    except Exception:
        skipped += 1
        continue

    # Collect all coefficients
    shop_meta = {
        "customer_code": shop,
        "n_rows": len(m),
        "r_squared": model.rsquared,
    }

    for term in model.params.index:
        coef = model.params[term]
        p = model.pvalues[term]
        lo, hi = model.conf_int().loc[term]

        se = model.bse[term]
        common_row = {
            **shop_meta,
            "effect_pct": np.expm1(coef),
            "ci_low_pct": np.expm1(lo),
            "ci_high_pct": np.expm1(hi),
            "coef": coef,
            "se": se,  # <-- add this
            "p_value": p,
        }

        if term.startswith("C(band)[") and "T." in term:
            shop_results.append(
                {
                    **common_row,
                    "variable": "rain",
                    "level": term.split("T.")[-1].rstrip("]"),
                }
            )
        elif term.startswith("C(month)[") and "T." in term:
            shop_results.append(
                {
                    **common_row,
                    "variable": "month",
                    "level": term.split("T.")[-1].rstrip("]"),
                }
            )
        elif term.startswith("C(dow)[") and "T." in term:
            shop_results.append(
                {
                    **common_row,
                    "variable": "dow",
                    "level": term.split("T.")[-1].rstrip("]"),
                }
            )
        elif term == "temperature":
            shop_results.append(
                {**common_row, "variable": "temperature", "level": "per_1C"}
            )
        elif term == "windspeed":
            shop_results.append(
                {**common_row, "variable": "windspeed", "level": "per_1_unit"}
            )

n_shops_fit = len(shops) - skipped
print(f"\nDone. Fitted: {n_shops_fit}  |  Skipped: {skipped}")

results_df = pd.DataFrame(shop_results)
print(f"Total rows in results_df: {len(results_df)}")
print(f"Variables collected: {results_df['variable'].unique().tolist()}")
results_df.head()

Running OLS for 613 shops ...


KeyboardInterrupt: 

In [9]:
print(sellout.shape)

sellout_dataaa = sellout.copy()

print(sellout_dataaa.shape)

(7859862, 20)
(7859862, 20)


In [24]:
# sellout = sellout_dataaa[sellout_dataaa["date"] <= "2024-12-31"].copy()
# Get 2025 data only.
sellout = sellout_dataaa[
    (sellout_dataaa["date"] >= "2024-01-01") & (sellout_dataaa["date"] <= "2025-12-31")
].copy()
print(sellout.shape)

(7063514, 20)


In [25]:
# ── Setup ─────────────────────────────────────────────────────────────────────
shop_results = []
skipped = 0
cur_shop = 1

common_shops = set(sellout["customer_code"].unique()) & set(
    sellin["customer_code"].unique()
)
shops = list(common_shops)
print(f"Running OLS for {len(shops)} shops ...")

RAIN_BINS = [-0.01, 0.1, 2, 8, 1e9]
RAIN_LABS = ["none", "light", "moderate", "heavy"]
MIN_ROWS = 30

# ── Per-shop loop ─────────────────────────────────────────────────────────────
for shop in shops:
    print(f"Shop # {cur_shop} / {len(shops)}", end="\r")
    cur_shop += 1

    # Prepare shop data
    sp = sellout[sellout["customer_code"] == shop].copy()
    sp.drop_duplicates(inplace=True)

    new_data = (
        sp.groupby("date")
        .agg(
            sales_quantity=("sales_quantity", "sum"),
            sales_amount=("sales_amount", "sum"),
            latitude=("latitude", "first"),
            longitude=("longitude", "first"),
        )
        .reset_index()
    )

    # Merge weather
    merged = new_data.merge(
        weather_df[
            [
                "date",
                "latitude",
                "longitude",
                "precipitation",
                "temperature",
                "windspeed",
            ]
        ],
        on=["date", "latitude", "longitude"],
        how="left",
    ).dropna(subset=["precipitation", "temperature", "windspeed"])

    # Quality filter
    m = merged[merged["sales_quantity"] > 0].copy()
    if len(m) < MIN_ROWS:
        skipped += 1
        continue

    # Features
    m = m.sort_values("date").reset_index(drop=True)
    m["log_q"] = np.log(m["sales_quantity"])
    m["dow"] = m["date"].dt.dayofweek.astype("category")
    m["month"] = m["date"].dt.month.astype("category")
    m["trend"] = (m["date"] - m["date"].min()).dt.days
    m["band"] = pd.Categorical(
        pd.cut(m["precipitation"], RAIN_BINS, labels=RAIN_LABS),
        categories=RAIN_LABS,
    )

    # Day-after rain band (only keep rows where prev row is the previous calendar day)
    m["prev_date"] = m["date"].shift(1)
    m["band_lag1"] = m["band"].shift(1)
    m = m[m["date"] - m["prev_date"] == pd.Timedelta(days=1)].copy()
    m = m.drop(columns=["prev_date"]).reset_index(drop=True)
    m["band_lag1"] = pd.Categorical(m["band_lag1"], categories=RAIN_LABS)

    # Re-check size after lag filter
    if len(m) < MIN_ROWS:
        skipped += 1
        continue

    # Need at least 2 levels in BOTH same-day and lag bands
    if m["band"].nunique(dropna=True) < 2 or m["band_lag1"].nunique(dropna=True) < 2:
        skipped += 1
        continue

    # Fit model
    formula = (
        "log_q ~ C(band) + C(band_lag1)"
        " + temperature + windspeed"
        " + C(dow) + C(month) + trend"
    )
    try:
        model = smf.ols(formula, data=m).fit(cov_type="HC3")
    except Exception:
        skipped += 1
        continue

    # Collect all coefficients
    shop_meta = {
        "customer_code": shop,
        "n_rows": len(m),
        "r_squared": model.rsquared,
    }

    for term in model.params.index:
        coef = model.params[term]
        p = model.pvalues[term]
        se = model.bse[term]
        lo, hi = model.conf_int().loc[term]

        common_row = {
            **shop_meta,
            "effect_pct": np.expm1(coef),
            "ci_low_pct": np.expm1(lo),
            "ci_high_pct": np.expm1(hi),
            "coef": coef,
            "se": se,
            "p_value": p,
        }

        if term.startswith("C(band)[") and "T." in term:
            shop_results.append(
                {
                    **common_row,
                    "variable": "rain",
                    "level": term.split("T.")[-1].rstrip("]"),
                }
            )
        elif term.startswith("C(band_lag1)[") and "T." in term:
            shop_results.append(
                {
                    **common_row,
                    "variable": "rain_lag1",
                    "level": term.split("T.")[-1].rstrip("]"),
                }
            )
        elif term.startswith("C(month)[") and "T." in term:
            shop_results.append(
                {
                    **common_row,
                    "variable": "month",
                    "level": term.split("T.")[-1].rstrip("]"),
                }
            )
        elif term.startswith("C(dow)[") and "T." in term:
            shop_results.append(
                {
                    **common_row,
                    "variable": "dow",
                    "level": term.split("T.")[-1].rstrip("]"),
                }
            )
        elif term == "temperature":
            shop_results.append(
                {**common_row, "variable": "temperature", "level": "per_1C"}
            )
        elif term == "windspeed":
            shop_results.append(
                {**common_row, "variable": "windspeed", "level": "per_1_unit"}
            )

n_shops_fit = len(shops) - skipped
print(f"\nDone. Fitted: {n_shops_fit}  |  Skipped: {skipped}")

results_df = pd.DataFrame(shop_results)
print(f"Total rows in results_df: {len(results_df)}")
print(f"Variables collected: {results_df['variable'].unique().tolist()}")
results_df.head()

Running OLS for 612 shops ...


/home/asad/anaconda3/envs/footfall_explorer/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:2014: RuntimeWarning: divide by zero encountered in divide
  self.het_scale = (self.wresid / (1 - h))**2


/tmp/ipykernel_13456/2122383186.py:115: RuntimeWarning: overflow encountered in expm1
  "ci_high_pct": np.expm1(hi),


/tmp/ipykernel_13456/2122383186.py:115: RuntimeWarning: overflow encountered in expm1
  "ci_high_pct": np.expm1(hi),


/tmp/ipykernel_13456/2122383186.py:115: RuntimeWarning: overflow encountered in expm1
  "ci_high_pct": np.expm1(hi),


/home/asad/anaconda3/envs/footfall_explorer/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:2014: RuntimeWarning: divide by zero encountered in divide
  self.het_scale = (self.wresid / (1 - h))**2


/tmp/ipykernel_13456/2122383186.py:115: RuntimeWarning: overflow encountered in expm1
  "ci_high_pct": np.expm1(hi),


Shop # 612 / 612
Done. Fitted: 551  |  Skipped: 61
Total rows in results_df: 13622
Variables collected: ['rain', 'rain_lag1', 'dow', 'month', 'temperature', 'windspeed']


,customer_code,n_rows,r_squared,effect_pct,ci_low_pct,ci_high_pct,coef,se,p_value,variable,level
0,0011t000011b1WsAAI,440,0.412011,0.025842,-0.066288,0.127063,0.025514,0.048012,0.595135,rain,light
1,0011t000011b1WsAAI,440,0.412011,0.019399,-0.078072,0.127175,0.019213,0.051277,0.707884,rain,moderate
2,0011t000011b1WsAAI,440,0.412011,0.014214,-0.096743,0.138801,0.014114,0.059115,0.811295,rain,heavy
3,0011t000011b1WsAAI,440,0.412011,-0.024386,-0.104567,0.062975,-0.024688,0.043756,0.572597,rain_lag1,light
4,0011t000011b1WsAAI,440,0.412011,-0.085417,-0.170143,0.007960,-0.089287,0.049600,0.071842,rain_lag1,moderate


In [112]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

MIN_ROWS = 30

# ── Define bin schemes to test ─────────────────────────────────────────────────
# Each scheme is (name, bin_edges, labels)
# Edges must be monotonically increasing; first edge below 0 to capture exact 0s.
BIN_SCHEMES = {
    "original": ([-0.01, 0.1, 2, 8, 1e9], ["none", "light", "moderate", "heavy"]),
    "tighter_light": ([-0.01, 0.1, 1, 5, 1e9], ["none", "light", "moderate", "heavy"]),
    "wmo_like": (
        [-0.01, 0.1, 2.5, 7.6, 1e9],
        ["none", "light", "moderate", "heavy"],
    ),  # WMO-ish daily thresholds
    "shifted_up": ([-0.01, 0.1, 3, 10, 1e9], ["none", "light", "moderate", "heavy"]),
    "quartile_style": ([-0.01, 0.1, 1, 3, 1e9], ["none", "light", "moderate", "heavy"]),
    "fine_grained": (
        [-0.01, 0.1, 1, 3, 8, 1e9],
        ["none", "light", "moderate", "heavy", "extreme"],
    ),
}

# ── One-time prep (unchanged from previous optimization) ──────────────────────
sellout_agg = (
    sellout.drop_duplicates()
    .groupby(["customer_code", "date"], as_index=False)
    .agg(
        sales_quantity=("sales_quantity", "sum"),
        sales_amount=("sales_amount", "sum"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
    )
)

weather_cols = [
    "date",
    "latitude",
    "longitude",
    "precipitation",
    "temperature",
    "windspeed",
]
all_merged = sellout_agg.merge(
    weather_df[weather_cols], on=["date", "latitude", "longitude"], how="left"
).dropna(subset=["precipitation", "temperature", "windspeed"])
all_merged = all_merged[all_merged["sales_quantity"] > 0].copy()
all_merged["log_q"] = np.log(all_merged["sales_quantity"])
all_merged["dow"] = all_merged["date"].dt.dayofweek
all_merged["month"] = all_merged["date"].dt.month

common_shops = set(sellout["customer_code"].unique()) & set(
    sellin["customer_code"].unique()
)


# ── Function: fit all shops for ONE bin scheme ────────────────────────────────
def fit_all_shops(scheme_name, bin_edges, labels, base_df):
    """Run the per-shop OLS loop for a given rain-bin scheme. Returns a DataFrame."""
    df = base_df.copy()
    df["band"] = pd.cut(df["precipitation"], bin_edges, labels=labels)

    grouped = df.sort_values(["customer_code", "date"]).groupby(
        "customer_code", sort=False
    )
    shops_to_fit = [s for s in grouped.groups if s in common_shops]

    results = []
    skipped = 0

    for i, shop in enumerate(shops_to_fit, 1):
        if i % 100 == 0:
            print(f"  [{scheme_name}] shop {i}/{len(shops_to_fit)}", end="\r")

        m = grouped.get_group(shop).reset_index(drop=True)
        if len(m) < MIN_ROWS:
            skipped += 1
            continue

        m["trend"] = (m["date"] - m["date"].iloc[0]).dt.days
        m["band_lag1"] = m["band"].shift(1)
        m = m[m["date"].diff() == pd.Timedelta(days=1)].reset_index(drop=True)
        if len(m) < MIN_ROWS:
            skipped += 1
            continue

        m["band"] = pd.Categorical(m["band"], categories=labels)
        m["band_lag1"] = pd.Categorical(m["band_lag1"], categories=labels)
        m["dow"] = m["dow"].astype("category")
        m["month"] = m["month"].astype("category")

        if (
            m["band"].nunique(dropna=True) < 2
            or m["band_lag1"].nunique(dropna=True) < 2
        ):
            skipped += 1
            continue

        formula = (
            "log_q ~ C(band) + C(band_lag1)"
            " + temperature + windspeed"
            " + C(dow) + C(month) + trend"
        )
        try:
            model = smf.ols(formula, data=m).fit(cov_type="HC1")
        except Exception:
            skipped += 1
            continue

        shop_meta = {
            "scheme": scheme_name,
            "customer_code": shop,
            "n_rows": len(m),
            "r_squared": model.rsquared,
        }
        params, pvals, ses, ci = (
            model.params,
            model.pvalues,
            model.bse,
            model.conf_int(),
        )

        for term in params.index:
            coef = params[term]
            lo, hi = ci.loc[term]
            common_row = {
                **shop_meta,
                "effect_pct": np.expm1(coef),
                "ci_low_pct": np.expm1(lo),
                "ci_high_pct": np.expm1(hi),
                "coef": coef,
                "se": ses[term],
                "p_value": pvals[term],
            }
            if term.startswith("C(band)[") and "T." in term:
                results.append(
                    {
                        **common_row,
                        "variable": "rain",
                        "level": term.split("T.")[-1].rstrip("]"),
                    }
                )
            elif term.startswith("C(band_lag1)[") and "T." in term:
                results.append(
                    {
                        **common_row,
                        "variable": "rain_lag1",
                        "level": term.split("T.")[-1].rstrip("]"),
                    }
                )
            elif term.startswith("C(month)[") and "T." in term:
                results.append(
                    {
                        **common_row,
                        "variable": "month",
                        "level": term.split("T.")[-1].rstrip("]"),
                    }
                )
            elif term.startswith("C(dow)[") and "T." in term:
                results.append(
                    {
                        **common_row,
                        "variable": "dow",
                        "level": term.split("T.")[-1].rstrip("]"),
                    }
                )
            elif term == "temperature":
                results.append(
                    {**common_row, "variable": "temperature", "level": "per_1C"}
                )
            elif term == "windspeed":
                results.append(
                    {**common_row, "variable": "windspeed", "level": "per_1_unit"}
                )

    print(
        f"  [{scheme_name}] fitted: {len(shops_to_fit) - skipped}  skipped: {skipped}"
    )
    return pd.DataFrame(results)


# ── Run every scheme and collect ──────────────────────────────────────────────
all_results = []
for scheme_name, (edges, labels) in BIN_SCHEMES.items():
    print(f"\n▶ Scheme: {scheme_name}  edges={edges}")
    df_scheme = fit_all_shops(scheme_name, edges, labels, all_merged)
    all_results.append(df_scheme)

results_all = pd.concat(all_results, ignore_index=True)
print(f"\nTotal rows across all schemes: {len(results_all)}")
print(f"Schemes tested: {results_all['scheme'].unique().tolist()}")


▶ Scheme: original  edges=[-0.01, 0.1, 2, 8, 1000000000.0]
  [original] fitted: 551  skipped: 1

▶ Scheme: tighter_light  edges=[-0.01, 0.1, 1, 5, 1000000000.0]
  [tighter_light] fitted: 551  skipped: 1

▶ Scheme: wmo_like  edges=[-0.01, 0.1, 2.5, 7.6, 1000000000.0]
  [wmo_like] fitted: 551  skipped: 1

▶ Scheme: shifted_up  edges=[-0.01, 0.1, 3, 10, 1000000000.0]
  [shifted_up] fitted: 551  skipped: 1

▶ Scheme: quartile_style  edges=[-0.01, 0.1, 1, 3, 1000000000.0]
  [quartile_style] fitted: 551  skipped: 1

▶ Scheme: fine_grained  edges=[-0.01, 0.1, 1, 3, 8, 1000000000.0]
  [fine_grained] fitted: 551  skipped: 1

Total rows across all schemes: 82006
Schemes tested: ['original', 'tighter_light', 'wmo_like', 'shifted_up', 'quartile_style', 'fine_grained']


In [110]:
def stars(p):
    if pd.isna(p):
        return ""
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else "ns"


def plot_weighted_effect(
    df,
    variable,
    levels,
    level_labels=None,
    title="",
    subtitle="inverse-variance weighted across shops",
    xaxis_title="",
    n_shops_label=None,
):
    """Weighted average effect across shops for any variable."""
    if level_labels is None:
        level_labels = [str(lv).capitalize() for lv in levels]

    means_pct, ci_lo_pct, ci_hi_pct, pvals, ns = [], [], [], [], []

    for lv in levels:
        sub = df[(df["variable"] == variable) & (df["level"] == lv)].dropna(
            subset=["coef", "se"]
        )
        sub = sub[sub["se"] > 0]
        n = len(sub)
        ns.append(n)

        if n == 0:
            means_pct.append(np.nan)
            ci_lo_pct.append(np.nan)
            ci_hi_pct.append(np.nan)
            pvals.append(np.nan)
            continue

        w = 1.0 / (sub["se"] ** 2)
        coef_bar = np.sum(w * sub["coef"]) / np.sum(w)
        se_bar = np.sqrt(1.0 / np.sum(w))

        z = coef_bar / se_bar
        p = 2 * (1 - scipy_stats.norm.cdf(abs(z)))
        lo = coef_bar - 1.96 * se_bar
        hi = coef_bar + 1.96 * se_bar

        means_pct.append(np.expm1(coef_bar))
        ci_lo_pct.append(np.expm1(lo))
        ci_hi_pct.append(np.expm1(hi))
        pvals.append(p)

    err_minus = [m - lo for m, lo in zip(means_pct, ci_lo_pct)]
    err_plus = [hi - m for m, hi in zip(means_pct, ci_hi_pct)]

    text_labels = [
        f"{v:+.1%}  {stars(p)}  (n={n})" for v, p, n in zip(means_pct, pvals, ns)
    ]
    colors = [
        "#c7395f" if (v is not None and not pd.isna(v) and v < 0) else "#2a9d8f"
        for v in means_pct
    ]

    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=list(level_labels),
            y=means_pct,
            marker_color=colors,
            error_y=dict(
                type="data",
                symmetric=False,
                array=err_plus,
                arrayminus=err_minus,
                color="#333",
                thickness=1.5,
                width=8,
            ),
            text=text_labels,
            textposition="outside",
            hovertemplate="<b>%{x}</b><br>effect: %{y:+.2%}<extra></extra>",
        )
    )
    fig.add_hline(y=0, line_color="gray", line_width=1)

    n_shops = (
        n_shops_label
        if n_shops_label is not None
        else df.loc[df["variable"] == variable, "customer_code"].nunique()
    )
    fig.update_layout(
        title=dict(
            text=f"<b>{title} — {n_shops} shops</b><br><sub>{subtitle}</sub>",
            x=0.02,
            xanchor="left",
        ),
        yaxis_title="Mean sales effect",
        yaxis_tickformat=".0%",
        xaxis_title=xaxis_title,
        template="plotly_white",
        showlegend=False,
        height=460,
        margin=dict(l=70, r=30, t=90, b=70),
    )
    return fig

In [111]:
# Same-day rain
fig_same = plot_weighted_effect(
    results_df,
    variable="rain",
    levels=["light", "moderate", "heavy"],
    title="Same-day effect of rain on sales (2024-2026)",
    xaxis_title="Rain intensity",
)
fig_same.show()

# Day-after rain
fig_next = plot_weighted_effect(
    results_df,
    variable="rain_lag1",
    levels=["light", "moderate", "heavy"],
    title="Day-after effect of rain on sales (2024-2026)",
    xaxis_title="Yesterday's rain intensity",
)
fig_next.show()

In [45]:
def stars(p):
    if pd.isna(p):
        return ""
    return "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.1 else "ns"


def plot_avg_effect(
    df,
    variable,
    levels,
    level_labels=None,
    title="",
    subtitle="",
    xaxis_title="",
    colors=None,
    n_shops_label=None,
):
    """Bar chart of average effect (across shops) for one variable."""
    if level_labels is None:
        level_labels = levels

    means, cis, pvals, ns = [], [], [], []
    for lv in levels:
        vals = df.loc[
            (df["variable"] == variable) & (df["level"] == lv), "effect_pct"
        ].dropna()
        n = len(vals)
        means.append(vals.mean() if n else np.nan)
        if n > 1:
            se = vals.sem()
            ci = scipy_stats.t.ppf(0.975, df=n - 1) * se
            pv = scipy_stats.ttest_1samp(vals, 0).pvalue
        else:
            ci, pv = 0, float("nan")
        cis.append(ci)
        pvals.append(pv)
        ns.append(n)

    text_labels = [f"{v:+.1%}  {stars(p)}" for v, p in zip(means, pvals)]

    if colors is None:
        colors = [
            "#c7395f" if (v is not None and not pd.isna(v) and v < 0) else "#2a9d8f"
            for v in means
        ]

    fig = go.Figure()
    fig.add_trace(
        go.Bar(
            x=level_labels,
            y=means,
            marker_color=colors,
            error_y=dict(
                type="data",
                symmetric=True,
                array=cis,
                color="#333",
                thickness=1.5,
                width=8,
            ),
            text=text_labels,
            textposition="outside",
            hovertemplate="<b>%{x}</b><br>avg effect: %{y:+.2%}<extra></extra>",
        )
    )
    fig.add_hline(y=0, line_color="gray", line_width=1)

    n_shops = (
        n_shops_label if n_shops_label is not None else df["customer_code"].nunique()
    )
    fig.update_layout(
        title=dict(
            text=f"<b>{title} — {n_shops} shops</b><br><sub>{subtitle}</sub>",
            x=0.02,
            xanchor="left",
        ),
        yaxis_title="Mean sales effect",
        yaxis_tickformat=".0%",
        xaxis_title=xaxis_title,
        template="plotly_white",
        showlegend=False,
        height=460,
        margin=dict(l=70, r=30, t=90, b=70),
    )
    return fig


# def plot_rain_effect(
#     df,
#     levels=("light", "moderate", "heavy"),
#     level_labels=None,
#     title="Effect of rain on sales",
#     subtitle="vs. no rain — inverse-variance weighted across shops",
#     n_shops_label=None,
# ):
#     """Weighted average rain effect across shops, with proper CI."""
#     if level_labels is None:
#         level_labels = [lv.capitalize() for lv in levels]

#     means_pct, ci_lo_pct, ci_hi_pct, pvals, ns = [], [], [], [], []

#     for lv in levels:
#         sub = df[(df["variable"] == "rain") & (df["level"] == lv)].dropna(
#             subset=["coef", "se"]
#         )
#         sub = sub[sub["se"] > 0]
#         n = len(sub)
#         ns.append(n)

#         if n == 0:
#             means_pct.append(np.nan)
#             ci_lo_pct.append(np.nan)
#             ci_hi_pct.append(np.nan)
#             pvals.append(np.nan)
#             continue

#         # Inverse-variance weights
#         w = 1.0 / (sub["se"] ** 2)
#         coef_bar = np.sum(w * sub["coef"]) / np.sum(w)
#         se_bar = np.sqrt(1.0 / np.sum(w))

#         # Wald test on log scale, then transform endpoints
#         z = coef_bar / se_bar
#         p = 2 * (1 - scipy_stats.norm.cdf(abs(z)))
#         lo = coef_bar - 1.96 * se_bar
#         hi = coef_bar + 1.96 * se_bar

#         means_pct.append(np.expm1(coef_bar))
#         ci_lo_pct.append(np.expm1(lo))
#         ci_hi_pct.append(np.expm1(hi))
#         pvals.append(p)

#     # Error bars as distance from the bar top
#     err_minus = [m - lo for m, lo in zip(means_pct, ci_lo_pct)]
#     err_plus = [hi - m for m, hi in zip(means_pct, ci_hi_pct)]

#     text_labels = [
#         f"{v:+.1%}  {stars(p)}  (n={n})" for v, p, n in zip(means_pct, pvals, ns)
#     ]
#     colors = [
#         "#c7395f" if (v is not None and not pd.isna(v) and v < 0) else "#2a9d8f"
#         for v in means_pct
#     ]

#     fig = go.Figure()
#     fig.add_trace(
#         go.Bar(
#             x=list(level_labels),
#             y=means_pct,
#             marker_color=colors,
#             error_y=dict(
#                 type="data",
#                 symmetric=False,
#                 array=err_plus,
#                 arrayminus=err_minus,
#                 color="#333",
#                 thickness=1.5,
#                 width=8,
#             ),
#             text=text_labels,
#             textposition="outside",
#             hovertemplate="<b>%{x}</b><br>effect: %{y:+.2%}<extra></extra>",
#         )
#     )
#     fig.add_hline(y=0, line_color="gray", line_width=1)

#     n_shops = (
#         n_shops_label
#         if n_shops_label is not None
#         else df.loc[df["variable"] == "rain", "customer_code"].nunique()
#     )
#     fig.update_layout(
#         title=dict(
#             text=f"<b>{title} — {n_shops} shops</b><br><sub>{subtitle}</sub>",
#             x=0.02,
#             xanchor="left",
#         ),
#         yaxis_title="Sales effect vs. no rain",
#         yaxis_tickformat=".0%",
#         xaxis_title="Rain intensity",
#         template="plotly_white",
#         showlegend=False,
#         height=460,
#         margin=dict(l=70, r=30, t=90, b=70),
#     )
#     return fig


def plot_distribution(df, variable, title="", subtitle="", xaxis_title=""):
    """Histogram of effect across shops for a continuous variable."""
    vals = df.loc[df["variable"] == variable, "effect_pct"].dropna()
    if len(vals) == 0:
        print(f"No data for {variable}")
        return None

    n = len(vals)
    mean_eff = vals.mean()
    median_eff = vals.median()
    pval = scipy_stats.ttest_1samp(vals, 0).pvalue if n > 1 else float("nan")

    fig = go.Figure()
    fig.add_trace(
        go.Histogram(
            x=vals * 100,
            nbinsx=40,
            marker_color="#5b9bd5",
            opacity=0.85,
            hovertemplate="Effect: %{x:+.2f}%<br>Shops: %{y}<extra></extra>",
        )
    )
    fig.add_vline(x=0, line_color="gray", line_width=1)
    fig.add_vline(
        x=mean_eff * 100,
        line_color="#c7395f",
        line_width=2,
        line_dash="dash",
        annotation_text=f"mean {mean_eff:+.2%}",
        annotation_position="top",
    )
    fig.add_vline(
        x=median_eff * 100,
        line_color="#1f4e79",
        line_width=2,
        line_dash="dot",
        annotation_text=f"median {median_eff:+.2%}",
        annotation_position="bottom",
    )

    fig.update_layout(
        title=dict(
            text=f"<b>{title} — {n} shops</b>"
            f"<br><sub>{subtitle}  ·  t-test p={pval:.3f} {stars(pval)}</sub>",
            x=0.02,
            xanchor="left",
        ),
        xaxis_title=xaxis_title,
        yaxis_title="Number of shops",
        template="plotly_white",
        showlegend=False,
        height=460,
        margin=dict(l=70, r=30, t=90, b=70),
    )
    return fig

In [46]:
rain_colors = ["#a8d0e6", "#5b9bd5", "#1f4e79"]

fig = plot_avg_effect(
    results_df,
    variable="rain",
    levels=["light", "moderate", "heavy"],
    title="Effect of same-day rain on sales",
    subtitle="Mean % effect by rainfall band vs dry days, averaged across shops (error bars = 95% CI)",
    xaxis_title="Rainfall band  (light: 0.1–2mm  ·  moderate: 2–8mm  ·  heavy: >8mm)",
    colors=rain_colors,
    n_shops_label=n_shops_fit,
)
fig.show()

In [59]:
months = [str(i) for i in range(2, 13)]
month_labels = [calendar.month_abbr[int(m)] for m in months]

fig = plot_avg_effect(
    results_df,
    variable="month",
    levels=months,
    level_labels=month_labels,
    title="Monthly seasonality on sales (vs January)",
    subtitle="Mean % effect by month, averaged across shops (error bars = 95% CI)",
    xaxis_title="Month",
    n_shops_label=n_shops_fit,
)
fig.show()

In [48]:
dow_levels = ["1", "2", "3", "4", "5", "6"]
dow_labels = ["Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

fig = plot_avg_effect(
    results_df,
    variable="dow",
    levels=dow_levels,
    level_labels=dow_labels,
    title="Day-of-week effect on sales (vs Monday)",
    subtitle="Mean % effect by day, averaged across shops (error bars = 95% CI)",
    xaxis_title="Day of week",
    n_shops_label=n_shops_fit,
)
fig.show()

In [49]:
# View A — single bar
fig = plot_avg_effect(
    results_df,
    variable="temperature",
    levels=["per_1C"],
    level_labels=["Per +1°C"],
    title="Temperature effect on sales — per additional °C",
    subtitle="Mean % effect of +1°C, averaged across shops (error bars = 95% CI)",
    xaxis_title="",
    n_shops_label=n_shops_fit,
)
fig.show()

# View B — distribution across shops
fig = plot_distribution(
    results_df,
    variable="temperature",
    title="Distribution of temperature effect across shops",
    subtitle="% change in sales per +1°C",
    xaxis_title="Effect of +1°C on sales (%)",
)
fig.show()

## Temperature affect

#### Single Shop

In [93]:
soww = sellout_dataaa.merge(
    weather_df, how="left", on=["date", "latitude", "longitude"]
)

In [55]:
# ── Single customer data ───────────────────────────────────────────────────────
sc = soww[soww["customer_code"] == "0011t000011b2hEAAQ"].copy()
sc["month_num"] = pd.to_datetime(sc["date"]).dt.month

CUSTOMER_ID = sc["customer_code"].iloc[0]
CUSTOMER_LAT = sc["latitude"].iloc[0]
CUSTOMER_LON = sc["longitude"].iloc[0]

print(f"Customer : {CUSTOMER_ID}")
print(f"Location : {CUSTOMER_LAT:.4f}, {CUSTOMER_LON:.4f}")
print(f"Date range: {sc['date'].min().date()} → {sc['date'].max().date()}")
print(f"Total rows: {len(sc):,}")
print(f"Unique days: {sc['date'].nunique()}")
print(f"Total sales qty : {sc['sales_quantity'].sum():,.0f}")
print(f"Total sales amt : {sc['sales_amount'].sum():,.0f}")

Customer : 0011t000011b2hEAAQ
Location : 48.8473, 2.2851
Date range: 2024-01-02 → 2026-03-31
Total rows: 31,979
Unique days: 813
Total sales qty : 130,444
Total sales amt : 2,177,176


In [65]:
# ── Aggregate to daily level ───────────────────────────────────────────────────
month_labels = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]


def assign_season(month):
    if month in (12, 1, 2):
        return "Winter"
    elif month in (3, 4, 5):
        return "Spring"
    elif month in (6, 7, 8):
        return "Summer"
    else:  # 9, 10, 11
        return "Autumn"


daily = (
    sc.groupby("date")
    .agg(
        sales_quantity=("sales_quantity", "sum"),
        sales_amount=("sales_amount", "sum"),
        temperature=("temperature", "mean"),
        precipitation=("precipitation", "mean"),
        windspeed=("windspeed", "mean"),
    )
    .reset_index()
    .sort_values("date")
)

daily["month_num"] = daily["date"].dt.month
daily["month_name"] = daily["month_num"].apply(lambda m: month_labels[m - 1])
daily["season"] = daily["month_num"].apply(assign_season)
daily["temp_band"] = daily["temperature"].apply(
    lambda t: (
        "Hot (>15°C)" if t > 15 else ("Mild (10–15°C)" if t > 10 else "Cold (≤10°C)")
    )
)

print(f"Daily rows : {len(daily):,}  (expected ~813)")
print(f"Date range : {daily['date'].min().date()} → {daily['date'].max().date()}")
print(f"Avg daily qty    : {daily['sales_quantity'].mean():.1f}")
print(f"Avg daily amount : {daily['sales_amount'].mean():.1f}")
print()
print(daily.head())

Daily rows : 813  (expected ~813)
Date range : 2024-01-02 → 2026-03-31
Avg daily qty    : 160.4
Avg daily amount : 2678.0

        date  sales_quantity  sales_amount  temperature  precipitation  \
0 2024-01-02           182.0        3070.0         10.8           14.0   
1 2024-01-03           152.0        2800.0          9.8            2.1   
2 2024-01-04           183.0        3150.0          8.8            1.3   
3 2024-01-05           199.0        3640.0          6.9            0.0   
4 2024-01-06           236.0        4330.0          4.8            1.6   

   windspeed  month_num month_name  season       temp_band  
0       41.2          1        Jan  Winter  Mild (10–15°C)  
1       32.8          1        Jan  Winter    Cold (≤10°C)  
2       28.3          1        Jan  Winter    Cold (≤10°C)  
3       25.2          1        Jan  Winter    Cold (≤10°C)  
4       22.2          1        Jan  Winter    Cold (≤10°C)  


In [66]:
# ── Monthly avg temperature (all years combined) ───────────────────────────────
monthly_temp = (
    daily.groupby("month_num")
    .agg(
        avg_temp=("temperature", "mean"),
        days=("date", "nunique"),
    )
    .reset_index()
    .round(2)
)
monthly_temp["month_name"] = monthly_temp["month_num"].apply(
    lambda m: month_labels[m - 1]
)
monthly_temp["season"] = monthly_temp["month_num"].apply(assign_season)

print(monthly_temp[["month_name", "avg_temp", "days", "season"]].to_string(index=False))

month_name  avg_temp  days season
       Jan      4.38    90 Winter
       Feb      7.42    85 Winter
       Mar      8.88    93 Spring
       Apr     12.09    60 Spring
       May     15.12    62 Spring
       Jun     19.07    58 Summer
       Jul     20.15    62 Summer
       Aug     20.64    62 Summer
       Sep     15.64    60 Autumn
       Oct     12.92    62 Autumn
       Nov      8.46    60 Autumn
       Dec      6.40    59 Winter


In [72]:
# ── Plot ───────────────────────────────────────────────────────────────────────
season_colors = {
    "Winter": "#4A90E2",  # blue
    "Spring": "#7ED321",  # green
    "Summer": "#F5A623",  # orange
    "Autumn": "#D0021B",  # red/brown
}

fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=monthly_temp["month_name"],
        y=monthly_temp["avg_temp"],
        marker_color=[season_colors[s] for s in monthly_temp["season"]],
        text=monthly_temp["avg_temp"].round(1).astype(str) + "°C",
        textposition="outside",
        showlegend=False,
        hovertemplate="<b>%{x}</b><br>Avg Temp: %{y:.1f}°C<br>Days: "
        + monthly_temp["days"].astype(str)
        + "<extra></extra>",
    )
)

fig.add_trace(
    go.Scatter(
        x=monthly_temp["month_name"],
        y=monthly_temp["avg_temp"],
        mode="lines+markers",
        line=dict(color="black", width=1.5),
        marker=dict(size=7),
        showlegend=False,
        hoverinfo="skip",
    )
)

for s, c in season_colors.items():
    fig.add_trace(go.Bar(x=[None], y=[None], marker_color=c, name=s))

fig.update_layout(
    title=f"Avg Monthly Temperature — Customer {CUSTOMER_ID} (all years combined)",
    template="plotly_white",
    height=420,
    yaxis_title="Temperature (°C)",
    xaxis_title="Month",
    legend=dict(orientation="h", y=-0.15),
    margin=dict(t=60, b=60),
)
fig.show()

In [73]:
# ── Monthly avg sales (all years combined) ─────────────────────────────────────
monthly_sales = (
    daily.groupby("month_num")
    .agg(
        avg_qty=("sales_quantity", "mean"),
        avg_amount=("sales_amount", "mean"),
        total_qty=("sales_quantity", "sum"),
        days=("date", "nunique"),
    )
    .reset_index()
    .round(2)
)
monthly_sales["month_name"] = monthly_sales["month_num"].apply(
    lambda m: month_labels[m - 1]
)
monthly_sales["season"] = monthly_sales["month_num"].apply(assign_season)

print(
    monthly_sales[
        ["month_name", "avg_qty", "avg_amount", "total_qty", "days", "season"]
    ].to_string(index=False)
)

month_name  avg_qty  avg_amount  total_qty  days season
       Jan   153.52     2734.96    13817.0    90 Winter
       Feb   156.64     2574.22    13314.0    85 Winter
       Mar    99.18     1523.40     9224.0    93 Spring
       Apr   174.43     2985.03    10466.0    60 Spring
       May   173.06     2930.48    10730.0    62 Spring
       Jun   174.93     3012.00    10146.0    58 Summer
       Jul   145.23     2410.05     9004.0    62 Summer
       Aug   183.29     3068.19    11364.0    62 Summer
       Sep   158.93     2600.68     9536.0    60 Autumn
       Oct   166.29     2691.74    10310.0    62 Autumn
       Nov   172.90     2822.78    10374.0    60 Autumn
       Dec   206.08     3442.54    12159.0    59 Winter


In [74]:
# ── Plot ───────────────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=("Avg Daily Sales Quantity", "Avg Daily Sales Amount"),
)

for row, col in [(1, "avg_qty"), (2, "avg_amount")]:
    fig.add_trace(
        go.Bar(
            x=monthly_sales["month_name"],
            y=monthly_sales[col],
            marker_color=[season_colors[s] for s in monthly_sales["season"]],
            text=monthly_sales[col].round(1),
            textposition="outside",
            showlegend=False,
            hovertemplate="<b>%{x}</b><br>%{y:.1f}<extra></extra>",
        ),
        row=row,
        col=1,
    )

    fig.add_trace(
        go.Scatter(
            x=monthly_sales["month_name"],
            y=monthly_sales[col],
            mode="lines+markers",
            line=dict(color="black", width=1.5),
            marker=dict(size=7),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=row,
        col=1,
    )

for s, c in season_colors.items():
    fig.add_trace(go.Bar(x=[None], y=[None], marker_color=c, name=s))

fig.update_layout(
    title=f"Avg Monthly Sales — Customer {CUSTOMER_ID} (all years combined)",
    template="plotly_white",
    height=620,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.08),
    margin=dict(t=80, b=60),
)
fig.update_yaxes(title_text="Avg Daily Qty", row=1, col=1)
fig.update_yaxes(title_text="Avg Daily Amount", row=2, col=1)
fig.show()

In [75]:
fig = make_subplots(specs=[[{"secondary_y": True}]])

# ── Sales bars (primary y) ─────────────────────────────────────────────────────
fig.add_trace(
    go.Bar(
        x=monthly_sales["month_name"],
        y=monthly_sales["avg_qty"],
        name="Avg Daily Qty",
        marker_color=[season_colors[s] for s in monthly_sales["season"]],
        opacity=0.8,
        hovertemplate="<b>%{x}</b><br>Avg Daily Qty: %{y:.1f}<extra></extra>",
    ),
    secondary_y=False,
)

# ── Temperature line (secondary y) ────────────────────────────────────────────
fig.add_trace(
    go.Scatter(
        x=monthly_temp["month_name"],
        y=monthly_temp["avg_temp"],
        name="Avg Temp (°C)",
        mode="lines+markers",
        line=dict(color="black", width=2.5, dash="dot"),
        marker=dict(size=8, color="black"),
        hovertemplate="<b>%{x}</b><br>Avg Temp: %{y:.1f}°C<extra></extra>",
    ),
    secondary_y=True,
)

fig.update_layout(
    title=f"Avg Daily Sales Qty vs Temperature — Customer {CUSTOMER_ID}",
    template="plotly_white",
    height=460,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.15),
    margin=dict(t=60, b=60),
)
fig.update_yaxes(title_text="Avg Daily Sales Quantity", secondary_y=False)
fig.update_yaxes(title_text="Temperature (°C)", secondary_y=True, showgrid=False)
fig.update_xaxes(title_text="Month")

fig.show()

#### On All Data

In [94]:
# ── Aggregate whole dataset to daily level ─────────────────────────────────────
# soww = soww[soww["Year"].isin([2024, 2025])]
daily_all = (
    soww.groupby("date")
    .agg(
        sales_quantity=("sales_quantity", "sum"),
        sales_amount=("sales_amount", "sum"),
        temperature=("temperature", "mean"),
        precipitation=("precipitation", "mean"),
        windspeed=("windspeed", "mean"),
    )
    .reset_index()
    .sort_values("date")
)

daily_all["month_num"] = daily_all["date"].dt.month
daily_all["month_name"] = daily_all["month_num"].apply(lambda m: month_labels[m - 1])
daily_all["season"] = daily_all["month_num"].apply(assign_season)
daily_all["temp_band"] = daily_all["temperature"].apply(
    lambda t: (
        "Hot (>15°C)" if t > 15 else ("Mild (10–15°C)" if t > 10 else "Cold (≤10°C)")
    )
)

print(f"Daily rows : {len(daily_all):,}")
print(
    f"Date range : {daily_all['date'].min().date()} → {daily_all['date'].max().date()}"
)
print(f"Avg daily qty    : {daily_all['sales_quantity'].mean():,.1f}")
print(f"Avg daily amount : {daily_all['sales_amount'].mean():,.1f}")

Daily rows : 821
Date range : 2024-01-01 → 2026-03-31
Avg daily qty    : 30,610.7
Avg daily amount : 429,418.1


In [95]:
# ── Monthly avg temperature + sales (all customers, all years) ─────────────────
monthly_all = (
    daily_all.groupby("month_num")
    .agg(
        avg_temp=("temperature", "mean"),
        avg_qty=("sales_quantity", "mean"),
        avg_amount=("sales_amount", "mean"),
        days=("date", "nunique"),
    )
    .reset_index()
    .round(2)
)
monthly_all["month_name"] = monthly_all["month_num"].apply(
    lambda m: month_labels[m - 1]
)
monthly_all["season"] = monthly_all["month_num"].apply(assign_season)

print(
    monthly_all[
        ["month_name", "avg_temp", "avg_qty", "avg_amount", "days", "season"]
    ].to_string(index=False)
)

month_name  avg_temp  avg_qty  avg_amount  days season
       Jan      4.39 28916.41   445071.33    93 Winter
       Feb      7.47 31959.47   476839.86    85 Winter
       Mar      8.94 28792.80   410134.13    93 Spring
       Apr     12.19 34399.77   465254.57    60 Spring
       May     15.20 33149.06   449609.68    62 Spring
       Jun     18.98 35376.98   481948.92    60 Summer
       Jul     20.22 29178.45   378143.92    62 Summer
       Aug     20.70 22418.31   303751.60    62 Summer
       Sep     15.73 31177.00   422746.90    60 Autumn
       Oct     12.97 32746.34   444128.69    62 Autumn
       Nov      8.50 29089.88   423618.48    60 Autumn
       Dec      6.34 31625.44   438440.90    62 Winter


In [96]:
# ── Plot: sales + temperature overlay ─────────────────────────────────────────
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=monthly_all["month_name"],
        y=monthly_all["avg_qty"],
        name="Avg Daily Qty",
        marker_color=[season_colors[s] for s in monthly_all["season"]],
        opacity=0.8,
        hovertemplate="<b>%{x}</b><br>Avg Daily Qty: %{y:,.1f}<extra></extra>",
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=monthly_all["month_name"],
        y=monthly_all["avg_temp"],
        name="Avg Temp (°C)",
        mode="lines+markers",
        line=dict(color="black", width=2.5, dash="dot"),
        marker=dict(size=8, color="black"),
        hovertemplate="<b>%{x}</b><br>Avg Temp: %{y:.1f}°C<extra></extra>",
    ),
    secondary_y=True,
)

for s, c in season_colors.items():
    fig.add_trace(go.Bar(x=[None], y=[None], marker_color=c, name=s))

fig.update_layout(
    title="Avg Daily Sales Qty vs Temperature — All Customers (all data)",
    template="plotly_white",
    height=460,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.15),
    margin=dict(t=60, b=60),
)
fig.update_yaxes(title_text="Avg Daily Sales Quantity", secondary_y=False)
fig.update_yaxes(title_text="Temperature (°C)", secondary_y=True, showgrid=False)
fig.update_xaxes(title_text="Month")
fig.show()

#### Check on all data. 


In [80]:
import pandas as pd
import numpy as np

# ── Average temperature by month (across all years) ───────────────────────────
sc["month_num"] = pd.to_datetime(sc["date"]).dt.month

monthly_temp = (
    sc.groupby("month_num")["temperature"]
    .mean()
    .reset_index()
    .rename(columns={"temperature": "avg_temp"})
)

month_labels = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "May",
    "Jun",
    "Jul",
    "Aug",
    "Sep",
    "Oct",
    "Nov",
    "Dec",
]
monthly_temp["month_name"] = monthly_temp["month_num"].apply(
    lambda m: month_labels[m - 1]
)


def assign_season(m):
    if m in [12, 1, 2]:
        return "Winter"
    elif m in [3, 4, 5]:
        return "Spring"
    elif m in [6, 7, 8]:
        return "Summer"
    else:
        return "Fall"


monthly_temp["season"] = monthly_temp["month_num"].apply(assign_season)

print("Average temperature by month (all years combined):\n")
print(monthly_temp[["month_name", "avg_temp", "season"]].to_string(index=False))

Average temperature by month (all years combined):

month_name  avg_temp season
       Jan  4.464441 Winter
       Feb  7.499075 Winter
       Mar  8.970293 Spring
       Apr 12.233018 Spring
       May 15.128043 Spring
       Jun 19.128874 Summer
       Jul 20.378453 Summer
       Aug 20.647589 Summer
       Sep 15.698255   Fall
       Oct 12.995313   Fall
       Nov  8.527473   Fall
       Dec  6.677144 Winter


In [81]:
# ── Plotly ────────────────────────────────────────────────────────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sc["season"] = sc["month_num"].apply(assign_season)

monthly_sales = (
    sc.groupby("month_num")
    .agg(avg_temp=("temperature", "mean"), avg_qty=("sales_quantity", "mean"))
    .reset_index()
)
monthly_sales["month_name"] = monthly_sales["month_num"].apply(
    lambda m: month_labels[m - 1]
)
monthly_sales["season"] = monthly_sales["month_num"].apply(assign_season)

season_colors = {
    "Winter": "#5b9bd5",
    "Spring": "#27ae60",
    "Summer": "#e74c3c",
    "Fall": "#e67e22",
}
transition_months = ["Mar", "Apr", "Oct", "Nov"]

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    subplot_titles=(
        "Average Temperature (°C) — all years",
        "Average Daily Sales Quantity — all years",
    ),
)

for row, col in [(1, "avg_temp"), (2, "avg_qty")]:
    fig.add_trace(
        go.Bar(
            x=monthly_sales["month_name"],
            y=monthly_sales[col],
            marker_color=[season_colors[s] for s in monthly_sales["season"]],
            text=monthly_sales[col].round(1),
            textposition="outside",
            name=col,
            showlegend=False,
            hovertemplate="<b>%{x}</b><br>Value: %{y:.2f}<extra></extra>",
        ),
        row=row,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=monthly_sales["month_name"],
            y=monthly_sales[col],
            mode="lines+markers",
            line=dict(color="black", width=1.5),
            marker=dict(size=6),
            showlegend=False,
            hoverinfo="skip",
        ),
        row=row,
        col=1,
    )

# Highlight transition months
for m in transition_months:
    for row in [1, 2]:
        fig.add_vrect(
            x0=m,
            x1=m,
            fillcolor="yellow",
            opacity=0.3,
            layer="below",
            line_width=20,
            row=row,
            col=1,
        )

for season, color in season_colors.items():
    fig.add_trace(go.Bar(x=[None], y=[None], marker_color=color, name=season))

fig.update_layout(
    height=650,
    title_text="Monthly Avg Temperature & Sales (All Years Combined)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.08, x=0),
    margin=dict(t=80, b=60),
)
fig.update_yaxes(title_text="Temperature (°C)", row=1, col=1)
fig.update_yaxes(title_text="Avg Daily Qty", row=2, col=1)
fig.update_xaxes(title_text="Month", row=2, col=1)

fig.show()

In [82]:
# ── Season summary ─────────────────────────────────────────────────────────────
season_order = ["Winter", "Spring", "Summer", "Fall"]

sales_by_season = (
    sc.groupby("season")
    .agg(
        total_quantity=("sales_quantity", "sum"),
        avg_daily_quantity=("sales_quantity", "mean"),
        total_amount=("sales_amount", "sum"),
        avg_temp=("temperature", "mean"),
        days=("date", "nunique"),
    )
    .round(2)
    .loc[season_order]
)
print("Sales by season:\n")
print(sales_by_season.to_string())

Sales by season:

        total_quantity  avg_daily_quantity  total_amount  avg_temp  days
season                                                                  
Winter       7366558.0                3.22   109106358.0      6.15   240
Spring       6796958.0                3.23    93933548.0     12.00   215
Summer       5321618.0                3.23    71194457.0     19.98   184
Fall         5646286.0                3.11    78317902.0     12.50   182


In [83]:
# ── Plotly: Sales by season ────────────────────────────────────────────────────
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=sales_by_season.index,
        y=sales_by_season["avg_daily_quantity"],
        marker_color=[season_colors[s] for s in season_order],
        text=sales_by_season["avg_daily_quantity"].round(2),
        textposition="outside",
        hovertemplate="<b>%{x}</b><br>Avg Daily Qty: %{y:.2f}<br>Avg Temp: "
        + sales_by_season["avg_temp"].round(1).astype(str)
        + "°C<extra></extra>",
    )
)

fig.update_layout(
    title="Average Daily Sales Quantity by Season",
    template="plotly_white",
    height=400,
    yaxis_title="Avg Daily Sales Quantity",
    xaxis_title="Season",
    showlegend=False,
    margin=dict(t=60, b=40),
)
fig.show()

In [84]:
# ── Hot vs Cold split ──────────────────────────────────────────────────────────
sc["temp_band"] = sc["temperature"].apply(
    lambda t: (
        "Hot (>15°C)" if t > 15 else ("Mild (10–15°C)" if t > 10 else "Cold (≤10°C)")
    )
)

hot_cold = (
    sc.groupby("temp_band")
    .agg(
        total_quantity=("sales_quantity", "sum"),
        avg_daily_quantity=("sales_quantity", "mean"),
        total_amount=("sales_amount", "sum"),
        avg_temp=("temperature", "mean"),
        days=("date", "nunique"),
    )
    .round(2)
    .loc[["Cold (≤10°C)", "Mild (10–15°C)", "Hot (>15°C)"]]
)
print("Hot vs Mild vs Cold:\n")
print(hot_cold.to_string())

Hot vs Mild vs Cold:

                total_quantity  avg_daily_quantity  total_amount  avg_temp  days
temp_band                                                                       
Cold (≤10°C)        10698894.0                3.17   160652615.0      5.81   821
Mild (10–15°C)       7443218.0                3.21   100302312.0     12.46   257
Hot (>15°C)          6989308.0                3.22    91597338.0     19.42   578


In [85]:
# ── Plotly: Hot vs Cold ────────────────────────────────────────────────────────
band_colors = {
    "Cold (≤10°C)": "#5b9bd5",
    "Mild (10–15°C)": "#27ae60",
    "Hot (>15°C)": "#e74c3c",
}

fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("Avg Daily Sales Quantity", "Total Sales Amount"),
    column_widths=[0.5, 0.5],
)

for col, metric in [(1, "avg_daily_quantity"), (2, "total_amount")]:
    fig.add_trace(
        go.Bar(
            x=hot_cold.index,
            y=hot_cold[metric],
            marker_color=[band_colors[b] for b in hot_cold.index],
            text=hot_cold[metric].round(1),
            textposition="outside",
            showlegend=False,
            hovertemplate="<b>%{x}</b><br>%{y:.2f}<extra></extra>",
        ),
        row=1,
        col=col,
    )

fig.update_layout(
    title="Sales Performance: Cold vs Mild vs Hot Days",
    template="plotly_white",
    height=420,
    margin=dict(t=80, b=40),
)
fig.show()

In [86]:
# ── Transition months deep-dive (Mar, Apr, Oct, Nov) ──────────────────────────
transition_nums = [3, 4, 10, 11]
transition_labels = ["Mar", "Apr", "Oct", "Nov"]

trans = (
    sc[sc["month_num"].isin(transition_nums)]
    .groupby("month_num")
    .agg(
        avg_temp=("temperature", "mean"),
        avg_daily_qty=("sales_quantity", "mean"),
        total_qty=("sales_quantity", "sum"),
        days=("date", "nunique"),
    )
    .round(2)
    .loc[transition_nums]
)
trans.index = transition_labels
trans["temp_change"] = trans["avg_temp"].diff().round(2)
trans["qty_change_pct"] = (trans["avg_daily_qty"].pct_change() * 100).round(1)

print("Transition months (all years combined):\n")
print(trans.to_string())

Transition months (all years combined):

     avg_temp  avg_daily_qty  total_qty  days  temp_change  qty_change_pct
Mar      8.97           3.15  2677730.0    93          NaN             NaN
Apr     12.23           3.30  2063986.0    60         3.26             4.8
Oct     13.00           3.15  2030273.0    62         0.77            -4.5
Nov      8.53           3.07  1745393.0    60        -4.47            -2.5


In [87]:
# ── Plotly: Transition months dual axis ───────────────────────────────────────
fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Bar(
        x=transition_labels,
        y=trans["avg_daily_qty"],
        name="Avg Daily Qty",
        marker_color=["#27ae60", "#27ae60", "#e67e22", "#e67e22"],
        opacity=0.75,
        hovertemplate="<b>%{x}</b><br>Avg Daily Qty: %{y:.2f}<extra></extra>",
    ),
    secondary_y=False,
)

fig.add_trace(
    go.Scatter(
        x=transition_labels,
        y=trans["avg_temp"],
        name="Avg Temp (°C)",
        mode="lines+markers",
        line=dict(color="#e74c3c", width=2.5),
        marker=dict(size=9),
        hovertemplate="<b>%{x}</b><br>Avg Temp: %{y:.1f}°C<extra></extra>",
    ),
    secondary_y=True,
)

fig.update_layout(
    title="Sales & Temperature in Transition Months (All Years)",
    template="plotly_white",
    height=420,
    hovermode="x unified",
    legend=dict(orientation="h", y=-0.15),
    margin=dict(t=60, b=60),
)
fig.update_yaxes(title_text="Avg Daily Sales Quantity", secondary_y=False)
fig.update_yaxes(title_text="Avg Temperature (°C)", secondary_y=True, showgrid=False)
fig.show()

In [88]:
sc.groupby(["temp_band", "brand_name"])["sales_amount"].sum().unstack().fillna(0)

brand_name,FR_BF_,FR_BF_ACCESSOIRES VYPE,FR_BF_AJJA 17,FR_BF_CRAVEN A,FR_BF_DUNHILL,FR_BF_E-LIQUIDES VYPE COOL,FR_BF_E-LIQUIDES VYPE CORE,FR_BF_E-LIQUIDES VYPE ESSENTIELS,FR_BF_E-LIQUIDES VYPE FRUIT,FR_BF_E-LIQUIDES VYPE GOURMAND,...,FR_BF_WINFIELD,FR_BF_ePEN 3,FR_BF_ePEN 3 CAPSULES COOL,FR_BF_ePEN 3 CAPSULES CORE,FR_BF_ePEN 3 CAPSULES FRUIT,FR_BF_ePEN 3 CAPSULES GOURMANDS,FR_BF_ePEN 3 CAPSULES SMOOTHIES,FR_BF_ePEN 3 CAPSULES VPRO,FR_BF_ePod CAPS,FR_BF_eTANK PRO
temp_band,,,,,,,,,,,,,,,,,,,,,
Cold (≤10°C),88504.0,0.0,146350.0,1556080.0,13567980.0,0.0,1.0,38.0,0.0,7.0,...,1378350.0,810.0,1.0,15.0,1.0,1.0,0.0,1.0,52.0,826.0
Hot (>15°C),12.0,0.0,90400.0,876240.0,7598980.0,1.0,1.0,40.0,1.0,13.0,...,862190.0,333.0,0.0,1.0,0.0,0.0,0.0,3.0,5.0,456.0
Mild (10–15°C),56027.0,1.0,102050.0,949140.0,8379500.0,0.0,2.0,18.0,8.0,18.0,...,891460.0,663.0,0.0,3.0,0.0,0.0,0.0,12.0,14.0,544.0


In [89]:
# ── Reshape and filter to top brands ──────────────────────────────────────────
brand_temp = (
    sc.groupby(["temp_band", "brand_name"])["sales_amount"].sum().unstack().fillna(0)
)

# Top 10 brands by total sales across all bands
top_brands = brand_temp.sum().sort_values(ascending=False).head(10).index.tolist()
brand_temp_top = brand_temp[top_brands]

# Clean brand names (remove "FR_BF_" prefix)
brand_temp_top.columns = brand_temp_top.columns.str.replace("FR_BF_", "", regex=False)

band_order = ["Cold (≤10°C)", "Mild (10–15°C)", "Hot (>15°C)"]
brand_temp_top = brand_temp_top.loc[band_order]

print("Top 10 brands by sales amount per temperature band:\n")
print(brand_temp_top.to_string())

Top 10 brands by sales amount per temperature band:

brand_name           VOGUE  LUCKY STRIKE     DUNHILL   ROTHMANS  PETER STUYVESANT  PALL MALL   CRAVEN A   WINFIELD      VELO  EPOD PRO
temp_band                                                                                                                             
Cold (≤10°C)    68113500.0    56358215.0  13567980.0  9290265.0         6140780.0  2124370.0  1556080.0  1378350.0  914807.0  316795.0
Mild (10–15°C)  43046440.0    34620235.0   8379500.0  5613535.0         3738500.0  1422730.0   949140.0   891460.0  795331.0  245871.0
Hot (>15°C)     39112620.0    32003665.0   7598980.0  5026325.0         3329100.0  1301630.0   876240.0   862190.0  773260.0  247193.0


In [90]:
# ── Plotly: Grouped bar — top brands by temp band ─────────────────────────────
import plotly.graph_objects as go

band_colors = {
    "Cold (≤10°C)": "#5b9bd5",
    "Mild (10–15°C)": "#27ae60",
    "Hot (>15°C)": "#e74c3c",
}

fig = go.Figure()

for band in band_order:
    fig.add_trace(
        go.Bar(
            name=band,
            x=brand_temp_top.columns,
            y=brand_temp_top.loc[band],
            marker_color=band_colors[band],
            hovertemplate="<b>%{x}</b><br>"
            + band
            + "<br>Sales: %{y:,.0f}<extra></extra>",
        )
    )

fig.update_layout(
    barmode="group",
    title="Top 10 Brands — Sales Amount by Temperature Band",
    template="plotly_white",
    height=480,
    xaxis_title="Brand",
    yaxis_title="Total Sales Amount",
    legend=dict(orientation="h", y=-0.2),
    margin=dict(t=60, b=80),
    xaxis_tickangle=-30,
)
fig.show()

## Stock Out Check

In [274]:
customer_code = "0011t000011b5ddAAA"
sku_code = "a0U3W000002BYaXUAW"
sellin_selected = sellin[
    (sellin["customer_code"] == customer_code) & (sellin["sku_code"] == sku_code)
]

sellout_selected = sellout[
    (sellout["customer_code"] == customer_code) & (sellout["sku_code"] == sku_code)
]

all_sku_selected_so = sellout[sellout["customer_code"] == customer_code]

all_sku_selected_so = all_sku_selected_so.groupby("date")["sales_quantity"].sum()
all_sku_selected_so = pd.DataFrame(
    {"date": all_sku_selected_so.index, "sales_quantity": all_sku_selected_so.values}
)


sellin_selected.shape, sellout_selected.shape

((61, 20), (793, 20))

In [275]:
sellin_selected = sellin_selected.groupby("date")["sales_quantity"].sum().reset_index()
sellout_selected = (
    sellout_selected.groupby("date")["sales_quantity"].sum().reset_index()
)

sellin_selected.shape, sellout_selected.shape

((59, 2), (793, 2))

In [276]:
# Get overall min and max date across both
min_date = min(sellin_selected["date"].min(), sellout_selected["date"].min())
max_date = max(sellin_selected["date"].max(), sellout_selected["date"].max())

full_date_range = pd.DataFrame({"date": pd.date_range(min_date, max_date, freq="D")})

sellin_selected = full_date_range.merge(sellin_selected, on="date", how="left").fillna(
    0
)
sellout_selected = full_date_range.merge(
    sellout_selected, on="date", how="left"
).fillna(0)

In [277]:
fig = go.Figure()

# Sell-out (orange/red line)
fig.add_trace(
    go.Scatter(
        x=sellout_selected["date"],
        y=sellout_selected["sales_quantity"],
        mode="lines+markers",
        name="Sell-out",
        line=dict(color="#E8472A", width=1.5),
        marker=dict(size=4),
    )
)

# Sell-in (purple line)
fig.add_trace(
    go.Scatter(
        x=sellin_selected["date"],
        y=sellin_selected["sales_quantity"],
        mode="lines+markers",
        name="Sell-in",
        line=dict(color="#7B1FA2", width=1.5),
        marker=dict(size=4),
    )
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Quantity",
    plot_bgcolor="white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    hovermode="x unified",
    xaxis=dict(showgrid=False, tickformat="%b %-d\n%Y"),
    yaxis=dict(gridcolor="#e0e0e0"),
)

fig.show()

### Full StockOut

In [278]:
delivery_dates = (
    sellin_selected[sellin_selected["sales_quantity"] > 0]["date"]
    .sort_values()
    .reset_index(drop=True)
)

sellout_indexed = sellout_selected.set_index("date")["sales_quantity"]
all_shop_indexed = all_sku_selected_so.set_index("date")["sales_quantity"]

stockout_periods = []

for i in range(1, len(delivery_dates)):
    prev_delivery = delivery_dates[i - 1]
    curr_delivery = delivery_dates[i]

    # Condition 1: day immediately before delivery must be 0
    check_date = curr_delivery - pd.Timedelta(days=1)
    if sellout_indexed.get(check_date, 0) != 0:
        continue

    # Condition 2: shop must have been open on that day
    # (if total shop sales = 0, the shop was closed, not a stockout)
    if all_shop_indexed.get(check_date, 0) == 0:
        continue

    # Condition 3: sellout on delivery date itself must be non-zero
    # if sellout_indexed.get(curr_delivery, 0) == 0:
    #     continue

    # Walk backwards to find where the zero streak started
    # (skipping closed days — they don't break the stockout streak)
    stockout_start = check_date
    while check_date >= prev_delivery:
        val = sellout_indexed.get(check_date, 0)
        shop_open = all_shop_indexed.get(check_date, 0) != 0

        if val != 0:
            # Real sale happened → stockout ended here
            stockout_start = check_date + pd.Timedelta(days=1)
            break

        if shop_open:
            # Shop was open but this SKU had 0 sales → still a stockout day
            stockout_start = check_date

        # If shop was closed, don't update stockout_start (closed days don't count)
        check_date -= pd.Timedelta(days=1)

    stockout_periods.append(
        {
            "stockout_start": stockout_start,
            "stockout_end": curr_delivery,
            "duration_days": (curr_delivery - stockout_start).days,
        }
    )

stockout_df = pd.DataFrame(stockout_periods)
print(f"Found {len(stockout_df)} stockout periods")
stockout_df

Found 1 stockout periods


,stockout_start,stockout_end,duration_days
0,2024-09-27,2024-09-30,3


### Open METEO Hourly data 

In [115]:
import requests
import pandas as pd

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 48.8566,  # Paris
    "longitude": 2.3522,
    "start_date": "2024-01-01",
    "end_date": "2026-05-15",
    "hourly": "precipitation,temperature_2m,windspeed_10m,relative_humidity_2m",
    "timezone": "Europe/Paris",
}

response = requests.get(url, params=params)
data = response.json()

df = pd.DataFrame(data["hourly"])
df["time"] = pd.to_datetime(df["time"])
print(df.head())
print(f"\nTotal rows: {len(df)}")  # should be ~744 (31 days × 24 hours)

                 time  precipitation  temperature_2m  windspeed_10m  \
0 2024-01-01 00:00:00            0.0             7.4           26.7   
1 2024-01-01 01:00:00            0.0             7.3           25.6   
2 2024-01-01 02:00:00            0.0             7.1           25.9   
3 2024-01-01 03:00:00            0.0             7.1           24.6   
4 2024-01-01 04:00:00            0.0             7.1           25.0   

   relative_humidity_2m  
0                    77  
1                    76  
2                    75  
3                    71  
4                    71  

Total rows: 20784


,time,precipitation,temperature_2m,windspeed_10m,relative_humidity_2m
0,2024-01-01 00:00:00,0.0,7.4,26.7,77
1,2024-01-01 01:00:00,0.0,7.3,25.6,76
2,2024-01-01 02:00:00,0.0,7.1,25.9,75
3,2024-01-01 03:00:00,0.0,7.1,24.6,71
4,2024-01-01 04:00:00,0.0,7.1,25.0,71
5,2024-01-01 05:00:00,0.0,6.8,24.5,72
6,2024-01-01 06:00:00,0.0,6.6,23.5,74
7,2024-01-01 07:00:00,0.0,6.4,22.9,75
8,2024-01-01 08:00:00,0.0,6.2,21.7,77
9,2024-01-01 09:00:00,0.0,6.1,19.5,79
